# 핵심어 기반 상품 이미지 대체텍스트 생성

이미지와 한국어 TF-IDF 핵심어를 모델에 입력해 1·2·3그램 조건별 대체텍스트를 생성한다.

| 모델 | 생성 언어 | 최종 문장 |
| --- | --- | --- |
| Qwen3-VL-2B-Instruct | 영어 | DeepL 한국어 번역 |
| HyperCLOVAX-SEED-Vision-Instruct-3B | 한국어 | 모델 생성 문장 |

이미지 단독 문장은 기존 CSV에서 가져온다. 기존 컬리 alt는 메타데이터에서 가져오며 비교용으로 사용한다.

필요한 데이터:

- `alt_preprocessed_5.csv`
- `n-gram_tf-idf/tfidf_1gram_ver2.csv`, `tfidf_2gram_ver2.csv`, `tfidf_3gram_ver2.csv`
- 각 카테고리의 `images/`와 `<category>_metadata.csv`

실행 방법:

1. 데이터가 있는 프로젝트 폴더에서 노트북을 열고 `PROJECT_ROOT`를 확인한다.
2. 실행 환경을 준비한 뒤 커널을 재시작하고 **Run All**을 실행한다.
3. DeepL API 키는 숨김 입력창 또는 `DEEPL_AUTH_KEY` 환경 변수로 입력한다.
4. 전체 데이터는 `TEST_LIMIT_PER_GROUP = None`, 일부 점검은 숫자로 설정한다.
5. 결과 폴더가 있으면 완료된 결과를 이어서 사용한다. 처음부터 생성하려면 `OUTPUT_DIR`을 새 폴더로 지정한다.

실험 설정과 입력 파일의 해시를 저장한다. 입력·프롬프트·모델·생성 설정을 변경하는 경우 새 결과 폴더를 사용한다.


## 1. 실행 환경

확인된 환경: PyTorch `2.7.1+cu128`, Transformers `4.57.1`, DeepL SDK `1.32.0`.
CUDA를 지원하는 PyTorch 환경을 사용한다. 필요한 패키지를 아래 셀에서 설치한 뒤 커널을 재시작한다.


In [1]:
# 최초 설치 시 아래 줄의 주석을 해제하고, 설치 후 커널 재시작.
# %pip install "transformers==4.57.1" "deepl==1.32.0" accelerate pandas pillow tqdm einops timm av decord


## 2. 경로와 실험 설정

이미지별 TF-IDF 상위 `TOP_K`개 단어·구를 사용한다. 기본값은 5개다.
동일한 구 수를 사용하므로 3그램은 1그램보다 입력 단어 수가 많다.


In [2]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter
from xml.sax.saxutils import escape as xml_escape
from datetime import datetime, timezone

import pandas as pd


PROJECT_ROOT = Path.cwd()  # 이미지와 입력 CSV가 있는 프로젝트 폴더


TFIDF_DIR = PROJECT_ROOT / "n-gram_tf-idf"
TFIDF_FILES = {n: TFIDF_DIR / f"tfidf_{n}gram_ver2.csv" for n in (1, 2, 3)}
BASELINE_CSV = PROJECT_ROOT / "alt_preprocessed_5.csv"

# 이미지 단독 비교에는 기존 CSV의 원문 열을 사용.
BASELINE_COLUMNS = {"qwen3": "qwen_3_deepl_KOR", "hyperclovax": "HyperCLOVAX_KOR"}

CATEGORY_MODELS = {
    "bakery": ["qwen3"],
    "convenience_meals": ["hyperclovax"],
    "fruits_nuts_rice": ["qwen3", "hyperclovax"],
    "vegetables": ["hyperclovax"],
}
MODEL_IDS = {
    "qwen3": "Qwen/Qwen3-VL-2B-Instruct",
    "hyperclovax": "naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B",
}
RUN_MODELS = ["qwen3", "hyperclovax"]

MODEL_REVISIONS = {
    "qwen3": "89644892e4d85e24eaac8bacfd4f463576704203",
    "hyperclovax": "a78900bdac94cabc4648fe42c9ed31c3f4d522bd",
}

TOP_K = 5
TEST_LIMIT_PER_GROUP = None  # 전체: None, 일부 점검: 이미지 수
INCLUDE_IMAGE_ONLY = True  # 기존 이미지 단독 문장 포함
MAX_NEW_TOKENS_BY_MODEL = {"qwen3": 150, "hyperclovax": 160}
RETRY_ERRORS = True
RETRY_TRANSLATION_ERRORS = True
TRANSLATE_QWEN_WITH_DEEPL = True  # False: 영어 생성까지만 실행
REQUIRE_METADATA = True

DEEPL_OPTIONS = {
    "source_lang": "EN", "target_lang": "KO",
    "tag_handling": "xml", "tag_handling_version": "v2",
    "ignore_tags": ["keep"], "non_splitting_tags": ["keep"],
}
NAME_PATTERN = re.compile(r'"[^"\r\n]+"|“[^”\r\n]+”')


# 완료된 결과를 이어서 사용. 새로 생성하려면 폴더명을 변경.
OUTPUT_DIR = PROJECT_ROOT / "keyword_alt_results_preprocessed5_final"
LONG_CSV = OUTPUT_DIR / "alt_keywords_long.csv"
COMPARE_CSV = OUTPUT_DIR / "alt_keywords_comparison.csv"
READABLE_CSV = OUTPUT_DIR / "alt_keywords_comparison_readable.csv"
VIEWER_HTML = OUTPUT_DIR / "alt_comparison_viewer.html"
MANIFEST_JSON = OUTPUT_DIR / "generation_config.json"
QWEN_TRANSLATIONS_CSV = OUTPUT_DIR / "qwen_english_deepl.csv"

KEYS = ["category", "model", "product_id", "image_order"]
IMAGE_KEYS = ["category", "product_id", "image_order"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

QWEN_PROMPT = (
    "Describe this image concisely. "
    "Provide the description in one line exactly as follows:\n"
    "English: <English description>"
)
QWEN_KEYWORD_CONTEXT = (
    "\n\nReference keywords (in Korean):\n{keywords}\n"
    "These are meaning hints, not a list of product or brand names. Use only hints that match the image. "
    "Write the description in English only. Do not provide a Korean translation. "
    "Only a name clearly legible in the image may be copied exactly in quotation marks; do not quote keyword phrases as names."
)
HYPER_PROMPT = (
    "시각장애인을 위한 웹 대체텍스트를 작성해 주세요. "
    "이미지에서 직접 확인되는 핵심 대상과 특징만 간결한 한국어 한 문장으로 설명해 주세요. "
    "보이지 않는 정보는 추측하지 마세요. "
    "'이미지에는', '사진은', '모습입니다', '있습니다' 같은 불필요한 표현은 되도록 사용하지 마세요. "
    "설명 문장만 출력해 주세요."
)
HYPER_KEYWORD_CONTEXT = (
    "\n\n참고 핵심어:\n{keywords}\n"
    "핵심어는 뜻을 참고하는 단어·구이며 상품명이나 상표명 목록이 아닙니다. "
    "이미지와 일치하는 내용만 참고해 자연스러운 한국어 문장으로 작성해 주세요. "
    "이미지에서 명확히 읽히는 이름만 원래 표기 그대로 따옴표에 넣고, 핵심어 조합을 이름으로 만들지 마세요."
)


def task_prompt(model_name, keywords):
    keyword_text = "\n".join("- " + str(word) for word in keywords)
    if model_name == "qwen3":
        context = QWEN_KEYWORD_CONTEXT.format(keywords=keyword_text) if keywords else ""
        return QWEN_PROMPT, context
    context = HYPER_KEYWORD_CONTEXT.format(keywords=keyword_text) if keywords else ""
    return HYPER_PROMPT, context

print("프로젝트 폴더:", PROJECT_ROOT)
print("TF-IDF 폴더:", TFIDF_DIR)
print("기존 이미지 단독 결과:", BASELINE_CSV, "확인됨" if BASELINE_CSV.is_file() else "이 폴더에 없음")
for n, path in TFIDF_FILES.items():
    print(path.name, ":", "확인됨" if path.is_file() else "이 폴더에 없음")
print("이미지별 핵심어 수:", TOP_K)
print("카테고리·모델별 이미지 제한:", TEST_LIMIT_PER_GROUP)
print("입력 핵심어: 한국어 / Qwen: 영어만 생성 → DeepL 한국어 / HyperCLOVAX: 한국어")
print("Qwen DeepL 번역:", TRANSLATE_QWEN_WITH_DEEPL)


프로젝트 폴더: d:\2026_09_27_alt
TF-IDF 폴더: d:\2026_09_27_alt\n-gram_tf-idf
기존 이미지 단독 결과: d:\2026_09_27_alt\alt_preprocessed_5.csv 확인됨
tfidf_1gram_ver2.csv : 확인됨
tfidf_2gram_ver2.csv : 확인됨
tfidf_3gram_ver2.csv : 확인됨
이미지별 핵심어 수: 5
카테고리·모델별 이미지 제한: None
입력 핵심어: 한국어 / Qwen: 영어만 생성 → DeepL 한국어 / HyperCLOVAX: 한국어
Qwen DeepL 번역: True


## 3. 입력 데이터 연결

TF-IDF 결과의 `n_gram_keyword`, `tfidf_score` 열을 사용한다.
이미지와 핵심어는 카테고리·모델·상품 ID·이미지 순서로 연결한다.
기존 이미지 단독 문장은 CSV 원문 열에서 읽고, 기존 컬리 alt는 메타데이터에서 읽는다.


In [3]:
def read_csv_text(path):
    if not path.is_file():
        if path.name in {f"tfidf_{n}gram_ver2.csv" for n in (1, 2, 3)}:
            visible_csv = sorted(p.name for p in path.parent.glob("*.csv")) if path.parent.is_dir() else []
            raise FileNotFoundError(
                f"CSV 파일을 찾지 못했음: {path}\n"
                f"이 폴더의 CSV: {', '.join(visible_csv[:20]) or '(없음)'}\n"
                "tfidf_1gram_ver2.csv, tfidf_2gram_ver2.csv, tfidf_3gram_ver2.csv를 n-gram_tf-idf 폴더에 저장.\n"
                "PROJECT_ROOT는 이미지 카테고리 폴더들이 있는 상위 폴더로 설정.\n"
                "CSV 폴더 위치가 다르면 TFIDF_DIR를 수정한 뒤 설정 셀부터 다시 실행."
            )
        raise FileNotFoundError(f"파일을 찾지 못했음: {path}")
    for encoding in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding=encoding)
            df.columns = df.columns.str.strip()
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 인코딩 확인이 필요함: {path}")


def normalize_keys(df, has_model=True):
    df = df.copy()
    if "image_order" not in df.columns and "image_position" in df.columns:
        df = df.rename(columns={"image_position": "image_order"})
    required = KEYS if has_model else IMAGE_KEYS
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"필요한 컬럼이 없음: {sorted(missing)}")
    for col in required:
        df[col] = df[col].astype(str).str.strip()
    df["product_id"] = df["product_id"].str.replace(r"^(\d+)\.0+$", r"\1", regex=True)
    df["image_order"] = df["image_order"].str.replace(r"^(\d+)\.0+$", r"\1", regex=True)
    for col in ("product_id", "image_order"):
        if not df[col].str.fullmatch(r"\d+").all():
            bad = df.loc[~df[col].str.fullmatch(r"\d+"), required].head(5)
            raise ValueError(f"{col} 값 확인이 필요함:\n{bad.to_string(index=False)}")
    df["image_order"] = df["image_order"].map(lambda x: str(int(x)))
    if has_model:
        aliases = {"qwen3": "qwen3", "qwen_3": "qwen3", "hyperclovax": "hyperclovax"}
        df["model"] = df["model"].str.lower().map(aliases)
        if df["model"].isna().any():
            raise ValueError("model 값은 qwen3 또는 hyperclovax여야 함.")
    return df


def load_tfidf_tables():
    tables = {}
    for n, path in TFIDF_FILES.items():
        df = normalize_keys(read_csv_text(path))
        required = set(KEYS + ["n_gram_keyword", "tfidf_score"])
        if not required.issubset(df.columns):
            raise ValueError(f"{path.name} 컬럼 확인이 필요함: {sorted(required - set(df.columns))}")
        df["n_gram_keyword"] = df["n_gram_keyword"].str.strip()
        df["tfidf_score"] = pd.to_numeric(df["tfidf_score"], errors="raise")
        if ((df["tfidf_score"] < 0) | ~df["tfidf_score"].map(lambda x: float('-inf') < x < float('inf'))).any():
            raise ValueError(f"{path.name}: TF-IDF 점수 확인이 필요함.")
        if (df["n_gram_keyword"] == "").any():
            raise ValueError(f"{path.name}: 빈 핵심어가 있음.")
        if df.duplicated(KEYS + ["n_gram_keyword"]).any():
            raise ValueError(f"{path.name}: 같은 이미지에 같은 핵심어가 중복됨.")
        for category, model_name in df[["category", "model"]].drop_duplicates().itertuples(index=False, name=None):
            if model_name not in CATEGORY_MODELS.get(category, []):
                raise ValueError(f"카테고리와 모델 조합 확인이 필요함: {category}, {model_name}")

        tables[n] = df.loc[df["tfidf_score"] > 0].copy()
    return tables


def scan_images():
    rows = []
    for category in CATEGORY_MODELS:
        folder = PROJECT_ROOT / category / "images"
        if not folder.is_dir():
            raise FileNotFoundError(f"이미지 폴더가 없음: {folder}")
        for path in sorted(folder.iterdir()):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            match = re.fullmatch(r"kurly_(\d+)_(\d+)", path.stem, flags=re.IGNORECASE)
            if not match:
                raise ValueError(f"이미지 파일명 규칙 확인이 필요함: {path.name}")
            rows.append({"category": category, "product_id": match.group(1),
                         "image_order": str(int(match.group(2))), "image_name": path.name,
                         "image_path": path.relative_to(PROJECT_ROOT).as_posix()})
    df = pd.DataFrame(rows, columns=IMAGE_KEYS + ["image_name", "image_path"])
    if df.empty:
        raise ValueError("이미지를 찾지 못했음.")
    if df.duplicated(IMAGE_KEYS).any():
        raise ValueError("같은 상품·이미지순서에 이미지 파일이 여러 개 있음.")
    return df.sort_values(IMAGE_KEYS, kind="stable").reset_index(drop=True)


def attach_original_alt(images):
    metadata = []
    for category in CATEGORY_MODELS:
        path = PROJECT_ROOT / category / f"{category}_metadata.csv"
        if not path.is_file():
            if REQUIRE_METADATA:
                raise FileNotFoundError(f"기존 alt 비교용 메타데이터가 없음: {path}")
            print("메타데이터 없음:", category)
            continue
        df = read_csv_text(path)
        df["category"] = category
        df = normalize_keys(df, has_model=False)
        if "original_alt" not in df.columns:
            raise ValueError(f"{path.name}에 original_alt 컬럼이 없음.")
        if df.duplicated(IMAGE_KEYS).any():
            raise ValueError(f"{path.name}에 같은 이미지 키가 중복됨.")
        metadata.append(df[IMAGE_KEYS + ["original_alt"]])
    if not metadata:
        return images.assign(prev_alt_KOR="", metadata_status="metadata_missing")
    meta = pd.concat(metadata, ignore_index=True).rename(columns={"original_alt": "prev_alt_KOR"})
    merged = images.merge(meta, on=IMAGE_KEYS, how="left", validate="one_to_one", indicator=True)
    unmatched = merged["_merge"] != "both"
    if REQUIRE_METADATA and unmatched.any():
        raise ValueError("메타데이터에 없는 이미지가 있음:\n" + merged.loc[unmatched, IMAGE_KEYS].head(5).to_string(index=False))
    merged["metadata_status"] = merged["_merge"].map({"both": "matched", "left_only": "metadata_missing", "right_only": ""}).astype(str)
    merged["prev_alt_KOR"] = merged["prev_alt_KOR"].fillna("")
    return merged.drop(columns="_merge")


def build_image_model_table(images, tables):
    rows = []
    for row in images.to_dict("records"):
        for model_name in CATEGORY_MODELS[row["category"]]:
            rows.append(dict(row, model=model_name))
    image_models = pd.DataFrame(rows)
    available = set(map(tuple, image_models[KEYS].to_numpy()))
    for n, df in tables.items():
        missing = set(map(tuple, df[KEYS].to_numpy())) - available
        if missing:
            raise ValueError(f"{n}그램 핵심어와 연결되는 이미지가 없음: {sorted(missing)[:5]}")
    return image_models.sort_values(KEYS, kind="stable").reset_index(drop=True)


def select_keywords(tables):
    selected = {}
    for n, df in tables.items():

        ranked = df.sort_values("tfidf_score", ascending=False, kind="stable")
        for key, group in ranked.groupby(KEYS, sort=False):
            top = group.head(TOP_K)
            selected[(tuple(key), n)] = {
                "keywords": top["n_gram_keyword"].tolist(),
                "scores": top["tfidf_score"].astype(float).tolist(),
            }
    return selected


def load_baseline():
    df = normalize_keys(read_csv_text(BASELINE_CSV), has_model=False)
    required = set(IMAGE_KEYS + list(BASELINE_COLUMNS.values()))
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{BASELINE_CSV.name}에 필요한 원문 컬럼이 없음: {sorted(missing)}")
    if df.duplicated(IMAGE_KEYS).any():
        raise ValueError(f"{BASELINE_CSV.name}에 같은 이미지가 중복됨.")
    return {tuple(row[col] for col in IMAGE_KEYS): row for row in df.to_dict("records")}


def normalize_qwen_english(english, keywords=()):
    """설명 전체를 감싼 따옴표를 제거하고 이름의 따옴표는 보존한다."""
    english = re.sub(r"\s+", " ", str(english)).strip()
    pairs = {'"': '"', '“': '”'}
    if len(english) < 3 or english[0] not in pairs or english[-1] != pairs[english[0]]:
        return english
    inside = english[1:-1].strip()
    candidates = {str(k).replace("_", " ").strip().strip('"“”').casefold() for k in keywords}
    if inside.casefold() in candidates:
        return english
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", inside)

    outside_inner_names = NAME_PATTERN.sub("", inside)
    predicate = re.search(
        r"\b(?:is|are|has|have|lies|lie|rests|rest|sits|sit|stands|stand|contains|contain|"
        r"shows|show|features|feature|holds|hold|displays|display|appears|appear)\b",
        outside_inner_names, flags=re.IGNORECASE,
    )
    sentence_end = inside.endswith((".", "!", "?"))
    subject_start = re.match(r"^(?:a|an|the|one|two|three|several|some|\d+)\b", inside,
                             flags=re.IGNORECASE)
    if len(words) >= 4 and (predicate or (subject_start and sentence_end)):
        return inside
    return english


def parse_qwen_english_only(raw):
    """영어 설명을 추출하고 별도의 한국어 출력은 오류로 처리한다."""
    text = str(raw).strip()
    if not text:
        raise ValueError("Qwen이 빈 문장을 출력했음.")

    if re.search(r"(?im)^\s*(?:\*\*)?(?:Korean|한국어)(?:\*\*)?\s*[:：]", text):
        raise ValueError("Qwen 영어만 생성 조건에서 한국어 번역 줄이 출력됨. raw_output 확인 필요.")
    english_labels = re.findall(r"(?im)^\s*(?:\*\*)?English(?:\*\*)?\s*[:：]", text)
    if len(english_labels) > 1:
        raise ValueError("Qwen 출력에 English 라벨이 중복됨. raw_output 확인 필요.")

    prefix = re.match(r"(?is)^\s*(?:\*\*)?English(?:\*\*)?\s*[:：]\s*(?:\*\*)?(.*)$", text)
    if prefix:
        text = prefix.group(1).strip()
    text = normalize_qwen_english(text)
    validate_qwen_english(text)
    return text

def protected_name_spans(text):
    """따옴표 안 이름과 인접한 짧은 한국어 괄호 표기를 찾는다."""
    spans = []
    covered_until = 0
    for match in NAME_PATTERN.finditer(text):
        if match.start() < covered_until:
            continue
        end = match.end()
        parenthetical = re.match(r"\s*\(([^()\r\n]+)\)", text[end:])
        if parenthetical:
            label = parenthetical.group(1).strip()
            short_label = len(label) <= 40 and len(label.split()) <= 4
            label_characters = re.fullmatch(r"[가-힣A-Za-z0-9_ &/+·-]+", label)
            sentence_ending = re.search(
                r"(?:습니다|입니다|합니다|된다|있다|이다|한다|보인다|어요|예요|에요)$", label,
            )
            if (short_label and label_characters and re.search(r"[가-힣]", label)
                    and not sentence_ending):
                end += parenthetical.end()
        spans.append((match.start(), end, text[match.start():end]))
        covered_until = end
    return spans


def outside_protected_names(text):
    parts, start = [], 0
    for left, right, _ in protected_name_spans(text):
        parts.append(text[start:left])
        start = right
    parts.append(text[start:])
    return "".join(parts)


def validate_qwen_english(english):
    english = normalize_qwen_english(english)
    spans = protected_name_spans(english)
    outside_names = outside_protected_names(english)
    if not re.search(r"[A-Za-z]", outside_names):
        raise ValueError("Qwen 출력에서 영어 설명을 확인하지 못했음. raw_output 확인 필요.")
    if re.search(r"[가-힣ㄱ-ㅎㅏ-ㅣ]", outside_names):
        raise ValueError("Qwen 영어 설명의 이름 표기 밖에 한국어가 출력됨. raw_output 확인 필요.")
    if any(re.search(r"\([^()]*[가-힣][^()]*\)", original) for _, _, original in spans):
        return "latin_text_with_parenthetical_labels_review"
    return "latin_text_with_quoted_names" if spans else "latin_text_present"


In [4]:
if isinstance(TOP_K, bool) or not isinstance(TOP_K, int) or TOP_K < 1:
    raise ValueError("TOP_K는 1 이상의 정수로 설정.")
if TEST_LIMIT_PER_GROUP is not None and (isinstance(TEST_LIMIT_PER_GROUP, bool) or not isinstance(TEST_LIMIT_PER_GROUP, int) or TEST_LIMIT_PER_GROUP < 1):
    raise ValueError("TEST_LIMIT_PER_GROUP은 양의 정수 또는 None으로 설정.")
if not RUN_MODELS or set(RUN_MODELS) - set(MODEL_IDS) or len(set(RUN_MODELS)) != len(RUN_MODELS):
    raise ValueError("RUN_MODELS 확인이 필요함.")

tfidf_tables = load_tfidf_tables()
image_table = attach_original_alt(scan_images())
image_model_table = build_image_model_table(image_table, tfidf_tables)
keyword_map = select_keywords(tfidf_tables)
baseline_map = load_baseline()
for row in image_model_table.to_dict("records"):
    image_key = tuple(row[col] for col in IMAGE_KEYS)
    column = BASELINE_COLUMNS[row["model"]]
    value = baseline_map.get(image_key, {}).get(column, "")
    if not value.strip() or value.strip().lower() == "error":
        raise ValueError(f"기존 이미지 단독 결과가 없거나 오류임: {image_key}, {column}")

coverage = image_model_table.groupby(["category", "model"]).size().rename("images").reset_index()
for n, df in tfidf_tables.items():
    counts = df[KEYS].drop_duplicates().groupby(["category", "model"]).size().rename(f"keywords_{n}gram")
    coverage = coverage.merge(counts.reset_index(), on=["category", "model"], how="left")
print(coverage.fillna(0).to_string(index=False))

sample_key = tuple(image_model_table.iloc[0][KEYS])
print("\n핵심어 연결 예시:", sample_key)
for n in TFIDF_FILES:
    print(f"{n}그램:", keyword_map.get((sample_key, n), {}).get("keywords", []))

selected_images = image_model_table[image_model_table["model"].isin(RUN_MODELS)]
if TEST_LIMIT_PER_GROUP is not None:
    selected_images = selected_images.groupby(["category", "model"], sort=False).head(TEST_LIMIT_PER_GROUP)
selected_images = selected_images.reset_index(drop=True)
print("\n이번 실행의 이미지·모델 조합 수:", len(selected_images))


         category       model  images  keywords_1gram  keywords_2gram  keywords_3gram
           bakery       qwen3     100             100             100             100
convenience_meals hyperclovax     100             100              98              93
 fruits_nuts_rice hyperclovax     100             100              99              92
 fruits_nuts_rice       qwen3     100             100             100             100
       vegetables hyperclovax     100             100              99              81

핵심어 연결 예시: ('bakery', 'qwen3', '1000128336', '1')
1그램: ['금빛', '듬뿍', '잘다', '호두', '체크']
2그램: ['가지다 빵', '금빛 갈색', '껍질 잘다', '덩이 체크', '들어가다 폭신폭신']
3그램: ['가지다 빵 덩이', '갈색 껍질 잘다', '금빛 갈색 껍질', '껍질 잘다 썰다', '덩이 체크 무늬']

이번 실행의 이미지·모델 조합 수: 500


## 4. 생성 조건과 저장 준비

이미지마다 1·2·3그램 조건을 만든다. 핵심어가 없으면 `no_keywords`로 기록한다.
기존 이미지 단독 문장은 `reused`로 기록한다. 재실행 시 완료된 결과를 유지하고 대기·오류 조건을 처리한다.


In [5]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value, path):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_csv(df, path):
    temp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)


def make_tasks(images):
    rows = []
    conditions = ([0] if INCLUDE_IMAGE_ONLY else []) + [1, 2, 3]
    for image in images.to_dict("records"):
        key = tuple(image[col] for col in KEYS)
        for n in conditions:
            kw = keyword_map.get((key, n), {"keywords": [], "scores": []})
            keywords = kw["keywords"]
            prompt, context = task_prompt(image["model"], keywords)
            baseline_column = BASELINE_COLUMNS[image["model"]] if n == 0 else ""
            baseline = baseline_map[tuple(image[col] for col in IMAGE_KEYS)][baseline_column] if n == 0 else ""
            row = dict(image, condition="image_only" if n == 0 else f"{n}gram", ngram=n,
                       model_id=MODEL_IDS[image["model"]], keywords=" / ".join(keywords),
                       keywords_json=json.dumps(keywords, ensure_ascii=False),
                       tfidf_scores_json=json.dumps(kw["scores"]), keyword_count=len(keywords),
                       keyword_word_count=sum(len(word.split()) for word in keywords),
                       prompt=(prompt if n != 0 else ""), keyword_context=context,
                       keyword_language="ko" if n else "",
                       keyword_source_file=TFIDF_FILES[n].name if n else "",
                       final_language="ko", english_check="", hit_token_limit="",
                       generated_alt=baseline, generated_alt_ENG="", generated_alt_KOR_direct="", raw_output="",
                       generation_status="reused" if n == 0 else ("pending" if keywords else "no_keywords"),
                       translation_status=("reused_translation" if n == 0 and image["model"] == "qwen3" else
                                           "not_required" if image["model"] == "hyperclovax" else "not_generated"),
                       generation_source=f"{BASELINE_CSV.name}:{baseline_column}" if n == 0 else "vlm",
                       translation_error="", translated_at="", deepl_sdk_version="", deepl_model_type_used="",
                       deepl_request_xml="", deepl_response_xml="", protected_names_json="", translation_method="",
                       error="" if n == 0 or keywords else "이 이미지·모델에 해당하는 핵심어가 없음.",
                       generated_token_count="",
                       model_revision="", generated_at="", elapsed_seconds="")
            rows.append(row)
    return pd.DataFrame(rows)


def comparison_table(long_df):
    fixed_cols = KEYS + ["model_id", "image_name", "image_path", "prev_alt_KOR", "metadata_status"]
    wide = long_df[fixed_cols].drop_duplicates(KEYS).copy()
    condition_names = (["image_only"] if INCLUDE_IMAGE_ONLY else []) + ["1gram", "2gram", "3gram"]
    for condition in condition_names:
        part = long_df.loc[long_df["condition"] == condition,
                           KEYS + ["keywords", "keyword_count", "generated_alt", "generated_alt_ENG", "generated_alt_KOR_direct",
                                   "generation_status", "translation_status", "generation_source", "english_check", "hit_token_limit", "error", "translation_error"]].copy()
        waiting = part["generation_status"].eq("ok") & part["translation_status"].eq("pending")
        part.loc[waiting, "generation_status"] = "translation_pending"
        failed = part["generation_status"].eq("ok") & part["translation_status"].eq("error")
        part.loc[failed, "generation_status"] = "translation_error"
        part.loc[failed, "error"] = part.loc[failed, "translation_error"]
        part = part.drop(columns="translation_error")
        part = part.rename(columns={"keywords": f"keywords_{condition}", "keyword_count": f"keyword_count_{condition}",
                                    "generated_alt": f"alt_{condition}", "generation_status": f"status_{condition}",
                                    "generated_alt_ENG": f"alt_ENG_{condition}",
                                    "generated_alt_KOR_direct": f"alt_KOR_direct_{condition}",
                                    "translation_status": f"translation_status_{condition}",
                                    "generation_source": f"source_{condition}",
                                    "english_check": f"english_check_{condition}", "hit_token_limit": f"hit_token_limit_{condition}",
                                    "error": f"error_{condition}"})
        wide = wide.merge(part, on=KEYS, how="left", validate="one_to_one")
    wide["all_ngram_ok"] = wide[[f"status_{n}gram" for n in (1, 2, 3)]].eq("ok").all(axis=1)
    return wide.sort_values(KEYS, kind="stable").reset_index(drop=True)


def save_results(df):
    ordered = df.sort_values(KEYS + ["ngram"], kind="stable").reset_index(drop=True)
    atomic_csv(ordered, LONG_CSV)
    atomic_csv(comparison_table(ordered), COMPARE_CSV)
    qwen = ordered.loc[(ordered["model"] == "qwen3") & ordered["condition"].ne("image_only"),
                       KEYS + ["condition", "generated_alt_ENG", "generated_alt", "generated_alt_KOR_direct",
                               "generation_status", "translation_status", "translation_error",
                               "deepl_request_xml", "deepl_response_xml", "protected_names_json", "translation_method"]].copy()
    qwen = qwen.rename(columns={"generated_alt_ENG": "source_english", "generated_alt": "deepl_korean",
                                "generated_alt_KOR_direct": "qwen_korean_direct"})
    atomic_csv(qwen, QWEN_TRANSLATIONS_CSV)


def prepare_results(current_tasks, config):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    existing_output = LONG_CSV.exists() or COMPARE_CSV.exists()
    if MANIFEST_JSON.exists():
        manifest = json.loads(MANIFEST_JSON.read_text(encoding="utf-8"))
        if manifest["experiment"] != config:
            raise ValueError("저장된 실험 설정과 현재 설정이 다름. OUTPUT_DIR을 새 폴더로 변경.")
    else:
        if existing_output:
            raise ValueError("결과 CSV는 있으나 설정 파일이 없음. OUTPUT_DIR을 새 폴더로 변경.")
        manifest = {"experiment": config, "created_at": datetime.now(timezone.utc).isoformat(), "models": {}}
        atomic_json(manifest, MANIFEST_JSON)
    if not LONG_CSV.exists():
        return current_tasks.copy(), manifest
    old = read_csv_text(LONG_CSV)
    identity = KEYS + ["condition"]
    if old.duplicated(identity).any():
        raise ValueError("저장된 결과에 이미지·모델·조건 중복이 있음.")
    current_ids = set(map(tuple, current_tasks[identity].to_numpy()))
    old_lookup = {tuple(row[c] for c in identity): row for row in old.to_dict("records")}
    result_cols = ["generated_alt", "generated_alt_ENG", "generated_alt_KOR_direct", "raw_output",
                   "generation_status", "translation_status", "generation_source", "error",
                   "translation_error", "translated_at", "deepl_sdk_version", "deepl_model_type_used",
                   "deepl_request_xml", "deepl_response_xml", "protected_names_json", "translation_method",
                   "generated_token_count", "english_check", "hit_token_limit", "model_revision", "generated_at", "elapsed_seconds"]
    rows = []
    for row in current_tasks.to_dict("records"):
        previous = old_lookup.get(tuple(row[c] for c in identity))
        if previous is not None:
            for col in result_cols:
                row[col] = previous.get(col, row.get(col, ""))
        rows.append(row)

    rows.extend(row for identity_key, row in old_lookup.items() if identity_key not in current_ids)
    return pd.DataFrame(rows, columns=current_tasks.columns), manifest


def recover_saved_qwen_outputs(df, current_tasks, manifest):
    """저장된 오류 출력에서 영어 설명을 다시 추출한다."""
    identity = KEYS + ["condition"]
    active_ids = set(map(tuple, current_tasks[identity].to_numpy()))
    candidates = df.index[df["model"].eq("qwen3") & df["condition"].ne("image_only")
                          & df["generation_status"].eq("error") & df["raw_output"].ne("")]
    events = []
    for i in candidates:
        if tuple(df.loc[i, identity]) not in active_ids:
            continue
        raw = df.at[i, "raw_output"]
        try:
            english = parse_qwen_english_only(raw)
            english = normalize_qwen_english(english, json.loads(df.at[i, "keywords_json"]))
            english_check = validate_qwen_english(english)
            revision = df.at[i, "model_revision"] or manifest.get("models", {}).get("qwen3", {}).get("resolved_revision", "")
            if not revision:
                continue
        except (ValueError, TypeError):
            continue
        events.append(dict(
            identity={key: str(df.at[i, key]) for key in identity},
            previous_error=str(df.at[i, "error"]),
            raw_sha256=hashlib.sha256(raw.encode("utf-8")).hexdigest(),
            recovered_at=datetime.now(timezone.utc).isoformat(),
        ))
        df.at[i, "generated_alt_ENG"] = english
        df.at[i, "generated_alt_KOR_direct"] = ""
        df.at[i, "english_check"] = english_check
        df.at[i, "generated_alt"] = ""
        df.at[i, "generation_status"] = "ok"
        df.at[i, "translation_status"] = "pending"
        df.at[i, "translation_error"] = ""
        df.at[i, "generation_source"] = "vlm + saved_output_recovery"
        df.at[i, "model_revision"] = revision
        df.at[i, "error"] = ""

    processing = manifest.setdefault("output_processing", {})
    processing["version"] = "qwen_english_parenthetical_labels_v2"
    if events:
        processing.setdefault("recoveries", []).extend(events)
    atomic_json(manifest, MANIFEST_JSON)
    print(f"저장된 Qwen 원문에서 복구: {len(events)}개 (이미지 재생성 없음)")
    return df


class NamePreservationError(ValueError):

    pass

def prepare_translation_xml(english):
    english = normalize_qwen_english(english)
    parts, names, start = [], [], 0
    for number, (left, right, original) in enumerate(protected_name_spans(english)):
        parts.append(xml_escape(english[start:left]))
        identity = f"n{number}"
        digest = hashlib.sha256((english + str(number)).encode("utf-8")).hexdigest()[:12]
        token = f"ALTNAME_{number}_{digest}"
        while token in english:
            token += "X"
        parts.append(f'<keep id="{identity}">{token}</keep>')
        names.append({"id": identity, "token": token, "original": original})
        start = right
    parts.append(xml_escape(english[start:]))
    return "<text>" + "".join(parts) + "</text>", names

def unpack_translation_xml(translated, expected_names):
    translated = str(translated).strip()


    translated = re.sub(r"^<\?xml[^?]*\?>\s*", "", translated)
    try:
        root = ET.fromstring("<translation>" + translated + "</translation>")
    except ET.ParseError:
        raise NamePreservationError("XML을 해석하지 못했음. deepl_response_xml 확인 필요.") from None
    if any(node.tag not in {"translation", "text", "keep"} for node in root.iter()):
        raise NamePreservationError("예상하지 않은 XML 태그가 반환됨. deepl_response_xml 확인 필요.")
    expected = {item["id"]: item for item in expected_names}
    if len(expected) != len(expected_names):
        raise NamePreservationError("보호할 이름 ID가 중복됨.")
    occurrences = list(root.iter("keep"))
    returned_ids = [node.get("id", "") for node in occurrences]
    if Counter(returned_ids) != Counter(expected.keys()):
        raise NamePreservationError(
            f"이름 위치 태그가 빠지거나 중복됨 (예상 {len(expected_names)}개, 반환 {len(occurrences)}개)."
        )
    for node in occurrences:
        identity = node.get("id")
        saved = expected[identity]
        content = (node.text or "").strip()
        token = saved["token"]

        if len(node) or content.count(token) != 1:
            raise NamePreservationError(f"이름 보호 표식 {identity}가 변경됨. deepl_response_xml 확인 필요.")
        prefix, suffix = content.split(token, 1)
        quote_chars = chr(34) + chr(39) + "“”‘’"
        quote_pattern = "[" + re.escape(quote_chars) + "]{0,3}"
        allowed_suffixes = {"", "은", "는", "이", "가", "을", "를", "의", "에", "에서",
                            "와", "과", "도", "만", "로", "으로", "라고", "이라고",
                            "라는", "이라는", "란", "이란"}
        particles = "|".join(re.escape(s) for s in sorted(allowed_suffixes, key=len, reverse=True))
        if not re.fullmatch(quote_pattern, prefix):
            raise NamePreservationError(f"이름 보호 표식 {identity} 앞에 예상하지 않은 내용이 붙음.")
        if not re.fullmatch(quote_pattern + "(?:" + particles + ")" + quote_pattern, suffix):
            raise NamePreservationError(f"이름 보호 표식 {identity} 뒤에 예상하지 않은 내용이 붙음.")

        node.text = prefix + saved["original"] + suffix
    korean = "".join(root.itertext()).strip()
    if not korean:
        raise ValueError("DeepL이 빈 번역을 반환했음.")
    if any(item["token"] in korean for item in expected_names):
        raise NamePreservationError("복원되지 않은 이름 보호 표식이 남음.")
    return korean

def prepare_retry_translation_xml(english, mode):
    """이름 원문 또는 짧은 표식으로 번역용 XML을 만든다."""
    if mode not in {"literal", "short_marker"}:
        raise ValueError("지원하지 않는 이름 보호 방식.")
    english = normalize_qwen_english(english)
    parts, names, start = [], [], 0
    for number, (left, right, original) in enumerate(protected_name_spans(english)):
        parts.append(xml_escape(english[start:left]))
        token = original if mode == "literal" else f"KEEP{number}X"
        if mode == "short_marker":
            while token in english:
                token += "X"
        identity = f"n{number}"
        parts.append(f'<keep id="{identity}">{xml_escape(token)}</keep>')
        names.append({"id": identity, "token": token, "original": original, "mode": mode})
        start = right
    parts.append(xml_escape(english[start:]))
    return "<text>" + "".join(parts) + "</text>", names



def unpack_retry_translation_xml(translated, expected_names):
    """이름 ID와 표식을 검증하고 한국어 설명을 복원한다."""
    translated = re.sub(r"^<\?xml[^?]*\?>\s*", "", str(translated).strip())
    try:
        root = ET.fromstring("<translation>" + translated + "</translation>")
    except ET.ParseError:
        raise NamePreservationError("반환 XML을 해석하지 못했음.") from None
    if any(node.tag not in {"translation", "text", "keep"} for node in root.iter()):
        raise NamePreservationError("예상하지 않은 XML 태그가 반환됨.")
    expected = {name["id"]: name for name in expected_names}
    if len(expected) != len(expected_names):
        raise NamePreservationError("보호할 이름 ID가 중복됨.")
    occurrences = list(root.iter("keep"))
    if Counter(node.get("id", "") for node in occurrences) != Counter(expected.keys()):
        raise NamePreservationError("이름 태그가 누락되거나 중복됨. 응답 일부를 성공 처리하지 않음.")

    outside_names = []
    for node in root.iter():
        if node.tag != "keep":
            outside_names.append(node.text or "")
        outside_names.append(node.tail or "")
    if not re.search(r"[가-힣]", "".join(outside_names)):
        raise NamePreservationError("이름 외의 한국어 설명을 확인하지 못했음.")
    # 짧은 표식은 태그 위치보다 실제 표식의 위치를 기준으로 복원한다.
    if expected_names and all(name.get("mode") == "short_marker" for name in expected_names):
        korean = "".join(root.itertext()).strip()
        if len({name["token"] for name in expected_names}) != len(expected_names):
            raise NamePreservationError("이름 표식이 중복됨.")
        patterns = []
        for name in expected_names:
            token = name["token"]
            pattern = re.compile(r"(?<![A-Za-z0-9_])" + re.escape(token) + r"(?![A-Za-z0-9_])")
            if not token or korean.count(token) != 1 or len(pattern.findall(korean)) != 1:
                raise NamePreservationError(f'이름 표식 {name["id"]}가 누락·변경되거나 중복됨.')
            patterns.append(pattern)
        for name, pattern in zip(expected_names, patterns):
            korean = pattern.sub(lambda match, original=name["original"]: original, korean, count=1)
        if any(name["token"] in korean for name in expected_names):
            raise NamePreservationError("복원되지 않은 이름 표식이 남음.")
        return korean

    for node in occurrences:
        saved = expected[node.get("id")]
        content, token = node.text or "", saved["token"]
        if len(node) or not token or content.count(token) != 1:
            raise NamePreservationError(f'이름 원문 또는 표식 {saved["id"]}가 변경됨.')


        prefix, suffix = content.split(token, 1)
        # 이름 주변에 추가된 조사·문구도 보존.
        node.text = prefix + saved["original"] + suffix
    korean = "".join(root.itertext()).strip()
    if not korean:
        raise NamePreservationError("DeepL이 빈 번역을 반환했음.")
    if any(name["mode"] == "short_marker" and name["token"] in korean
           for name in expected_names):
        raise NamePreservationError("복원되지 않은 이름 표식이 남음.")
    return korean



def recover_saved_short_marker_xml(df, current_tasks, manifest):
    """저장된 응답에서 태그 밖으로 이동한 정확한 이름 표식을 복원한다."""
    identity = KEYS + ["condition"]
    active_ids = set(map(tuple, current_tasks[identity].to_numpy()))
    candidates = df.index[df["model"].eq("qwen3") & df["condition"].ne("image_only")
                          & df["generation_status"].eq("ok") & df["translation_status"].eq("error")
                          & df["deepl_response_xml"].ne("")]
    recoveries = []
    for i in candidates:
        if tuple(df.loc[i, identity]) not in active_ids:
            continue
        try:
            payload, names = prepare_retry_translation_xml(df.at[i, "generated_alt_ENG"], "short_marker")
            if payload != df.at[i, "deepl_request_xml"] or names != json.loads(df.at[i, "protected_names_json"]):
                continue
            translated = unpack_retry_translation_xml(df.at[i, "deepl_response_xml"], names)
        except (ValueError, TypeError, KeyError):
            continue
        recoveries.append((i, translated))
    if not recoveries:
        print("저장된 이름 표식 위치 복구: 0개")
        return df
    import shutil
    backup_dir = OUTPUT_DIR / "backup_before_name_position_fix_v6"
    backup_dir.mkdir(parents=True, exist_ok=True)
    for path in (LONG_CSV, MANIFEST_JSON, COMPARE_CSV, READABLE_CSV, QWEN_TRANSLATIONS_CSV, VIEWER_HTML):
        destination = backup_dir / path.name
        if path.is_file() and not destination.exists():
            shutil.copy2(path, destination)
    events = []
    for i, translated in recoveries:
        events.append({
            "identity": {key: str(df.at[i, key]) for key in identity},
            "previous_error": str(df.at[i, "translation_error"]),
            "response_sha256": hashlib.sha256(df.at[i, "deepl_response_xml"].encode("utf-8")).hexdigest(),
            "recovered_at": datetime.now(timezone.utc).isoformat(),
        })
        df.at[i, "generated_alt"] = translated
        df.at[i, "translation_status"] = "translated"
        df.at[i, "translation_error"] = ""
        df.at[i, "translation_method"] = "quoted_labels_xml_short_marker_v6_saved_response"
        df.at[i, "generation_source"] = "vlm + deepl_saved_response_recovery"
    processing = manifest.setdefault("output_processing", {})
    processing["short_marker_position_recovery"] = {
        "version": "v6_exact_markers_in_complete_xml_text", "backup_directory": backup_dir.name,
    }
    processing.setdefault("short_marker_recoveries", []).extend(events)
    atomic_json(manifest, MANIFEST_JSON)
    print(f"저장된 이름 표식 위치 복구: {len(recoveries)}개 (모델 생성·DeepL 호출 없음)")
    return df


def recover_saved_deepl_xml(df, current_tasks, manifest):
    """저장된 번역 응답의 이름 표식을 검증하고 복원한다."""
    identity = KEYS + ["condition"]
    active_ids = set(map(tuple, current_tasks[identity].to_numpy()))
    candidates = df.index[df["model"].eq("qwen3") & df["condition"].ne("image_only")
                          & df["generation_status"].eq("ok") & df["translation_status"].eq("error")
                          & df["deepl_response_xml"].ne("")]
    events = []
    for i in candidates:
        if tuple(df.loc[i, identity]) not in active_ids:
            continue
        try:
            english = df.at[i, "generated_alt_ENG"]
            payload, expected_names = prepare_translation_xml(english)
            saved_names = json.loads(df.at[i, "protected_names_json"])
            if payload != df.at[i, "deepl_request_xml"] or saved_names != expected_names:
                raise NamePreservationError("저장된 이름 보호 정보가 영어 원문과 일치하지 않음.")
            translated = unpack_translation_xml(df.at[i, "deepl_response_xml"], expected_names)
        except (ValueError, TypeError, KeyError):
            continue
        events.append({
            "identity": {key: str(df.at[i, key]) for key in identity},
            "previous_error": str(df.at[i, "translation_error"]),
            "response_sha256": hashlib.sha256(df.at[i, "deepl_response_xml"].encode("utf-8")).hexdigest(),
            "recovered_at": datetime.now(timezone.utc).isoformat(),
        })
        df.at[i, "generated_alt"] = translated
        df.at[i, "translation_status"] = "translated"
        df.at[i, "translation_error"] = ""
        df.at[i, "translation_method"] = "quoted_labels_xml_id_placeholders_quotes_suffixes_v4_saved_response"
        df.at[i, "generation_source"] = "vlm + deepl_saved_response_recovery"

    processing = manifest.setdefault("output_processing", {})
    processing["name_recovery_version"] = "v4_exact_marker_quotes_and_korean_suffixes"
    if events:
        processing.setdefault("deepl_recoveries", []).extend(events)
    atomic_json(manifest, MANIFEST_JSON)
    print(f"저장된 DeepL 응답에서 복구: {len(events)}개 (API 재호출 없음)")
    return df

def backup_before_name_fix(manifest):
    """기존 결과와 설정을 한 번 백업한다."""
    processing = manifest.get("output_processing", {})
    if not LONG_CSV.exists() or processing.get("name_recovery_version") == "v4_exact_marker_quotes_and_korean_suffixes":
        return
    import shutil
    backup_dir = OUTPUT_DIR / "backup_before_name_fix_v4"
    backup_dir.mkdir(exist_ok=True)
    for path in (LONG_CSV, MANIFEST_JSON):
        target = backup_dir / path.name
        if path.is_file() and not target.exists():
            shutil.copy2(path, target)
    manifest.setdefault("output_processing", {})["backup_directory"] = backup_dir.name

image_hashes = {row.image_path: sha256_file(PROJECT_ROOT / row.image_path)
                for row in image_table.itertuples(index=False)}
experiment_config = {
    "schema_version": 7, "inference_version": "korean_keywords_ver2_qwen_english_line_deepl_suffixes", "top_k": TOP_K,
    "include_image_only": INCLUDE_IMAGE_ONLY, "image_only_source": "existing_file",
    "max_new_tokens_by_model": MAX_NEW_TOKENS_BY_MODEL, "do_sample": False,
    "generation_arguments": {name: {("max_length" if name == "hyperclovax" else "max_new_tokens"): MAX_NEW_TOKENS_BY_MODEL[name],
                                     "do_sample": False} for name in MODEL_IDS},
    "qwen_prompt": QWEN_PROMPT, "qwen_keyword_context": QWEN_KEYWORD_CONTEXT,
    "hyper_prompt": HYPER_PROMPT, "hyper_keyword_context": HYPER_KEYWORD_CONTEXT, "category_models": CATEGORY_MODELS,
    "qwen_final_caption": "english_to_korean_deepl_api", "deepl_options": DEEPL_OPTIONS,
    "dtype_by_model": {"qwen3": "torch.float16", "hyperclovax": "bf16_if_supported_else_fp16"},
    "baseline_sha256": sha256_file(BASELINE_CSV), "baseline_file": BASELINE_CSV.name,
    "keyword_language": "ko", "keyword_display": "plain_bullets_without_json_quotes", "qwen_generation_lines": ["en"], "hyper_generation_language": "ko",
    "final_language": "ko", "name_preservation": {"method": "quoted_spans_xml_id_placeholders_suffixes_v3", "pattern": NAME_PATTERN.pattern}, "baseline_columns": BASELINE_COLUMNS,
    "model_ids": MODEL_IDS, "requested_revisions": MODEL_REVISIONS,
    "tfidf_sha256": {str(n): sha256_file(path) for n, path in TFIDF_FILES.items()},
    "tfidf_files": {str(n): path.name for n, path in TFIDF_FILES.items()},
    "images_sha256": image_hashes,
    "metadata_sha256": {category: sha256_file(PROJECT_ROOT / category / f"{category}_metadata.csv")
                         if (PROJECT_ROOT / category / f"{category}_metadata.csv").is_file() else None
                         for category in CATEGORY_MODELS},
}

tasks = make_tasks(selected_images)
result_df, manifest = prepare_results(tasks, experiment_config)
backup_before_name_fix(manifest)
result_df = recover_saved_qwen_outputs(result_df, tasks, manifest)
result_df = recover_saved_deepl_xml(result_df, tasks, manifest)
result_df = recover_saved_short_marker_xml(result_df, tasks, manifest)
save_results(result_df)
print("조건별 상태:")
print(result_df.groupby(["model", "condition", "generation_status"]).size().to_string())
print("\n비교표:", COMPARE_CSV)


저장된 Qwen 원문에서 복구: 0개 (이미지 재생성 없음)
저장된 DeepL 응답에서 복구: 0개 (API 재호출 없음)
저장된 이름 표식 위치 복구: 1개 (모델 생성·DeepL 호출 없음)
조건별 상태:
model        condition   generation_status
hyperclovax  1gram       ok                   300
             2gram       no_keywords            4
                         ok                   296
             3gram       no_keywords           34
                         ok                   266
             image_only  reused               300
qwen3        1gram       ok                   200
             2gram       ok                   200
             3gram       ok                   200
             image_only  reused               200

비교표: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\alt_keywords_comparison.csv


## 5. DeepL API 연결

Qwen 영어 문장을 한국어로 번역한다. API 키는 파일에 저장하지 않는다.
`TRANSLATE_QWEN_WITH_DEEPL = False`이면 영어 생성까지만 실행한다.


In [6]:
import os
import getpass
from tqdm.auto import tqdm


def needs_deepl(df):
    if not TRANSLATE_QWEN_WITH_DEEPL:
        return False
    active_ids = set(map(tuple, tasks[KEYS + ["condition"]].to_numpy()))
    generation_states = ["pending", "error"] if RETRY_ERRORS else ["pending"]
    translation_states = ["pending", "error"] if RETRY_TRANSLATION_ERRORS else ["pending"]
    for row in df.to_dict("records"):
        if row["model"] != "qwen3" or row["condition"] == "image_only":
            continue
        if tuple(row[c] for c in KEYS + ["condition"]) not in active_ids:
            continue
        if row["generation_status"] in generation_states:
            return True
        if row["generation_status"] == "ok" and row["translation_status"] in translation_states:
            return True
    return False


def initialize_deepl():
    if not needs_deepl(result_df):
        print("Qwen 번역 비활성 또는 신규 번역 없음. DeepL 연결 생략.")
        return None
    try:
        import deepl
    except ImportError:
        raise ImportError('DeepL 패키지 설치 필요: %pip install "deepl>=1.30,<2" 후 커널 재시작.') from None
    sdk_version = importlib.metadata.version("deepl")
    info = {"provider": "DeepL API", "sdk_version": sdk_version, "options": DEEPL_OPTIONS}
    previous = manifest.get("translator")
    if previous is not None and previous != info:
        raise ValueError("저장된 DeepL 버전·번역 설정과 다름. 기존 환경을 사용하거나 OUTPUT_DIR을 새 폴더로 변경.")
    auth_key = os.environ.get("DEEPL_AUTH_KEY", "").strip()
    if not auth_key:
        auth_key = getpass.getpass("DeepL API 키 입력 (화면·파일에 저장하지 않음): ").strip()
    if not auth_key:
        raise ValueError("DeepL API 키가 비어 있음. 이 셀을 다시 실행해 입력.")
    try:
        client = deepl.DeepLClient(auth_key)
        usage = client.get_usage()
    except Exception as error:
        raise RuntimeError(
            f"DeepL 연결 확인 실패 ({type(error).__name__}). API 키·계정 상태·네트워크 및 사용량 조회 권한 확인 필요."
        ) from None
    if usage.any_limit_reached:
        raise RuntimeError("DeepL 번역 한도에 도달했음. 계정의 문자 한도 확인 필요.")
    manifest["translator"] = info
    atomic_json(manifest, MANIFEST_JSON)
    print("DeepL 연결 확인 완료:", sdk_version, "영어 → 한국어")
    return client


deepl_client = initialize_deepl()


Qwen 번역 비활성 또는 신규 번역 없음. DeepL 연결 생략.


## 6. 모델과 생성 함수

모델별 프롬프트·생성 상한·출력 언어를 고정하고 조건별 핵심어 목록을 바꾼다.
Qwen은 새 토큰 150개, HyperCLOVAX는 새 토큰 160개를 상한으로 사용한다.
원본 출력과 생성 토큰 수를 상세 CSV에 저장한다.


In [7]:
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor

import sys
print("실행 Python:", sys.executable)
print("Transformers 버전:", transformers.__version__)
if "qwen3" in RUN_MODELS:
    try:
        from transformers import Qwen3VLForConditionalGeneration
    except ImportError as error:
        raise ImportError(
            f"현재 Transformers {transformers.__version__}에서 Qwen3-VL 클래스를 불러오지 못했음.\n"
            "맨 위 설치 셀을 실행해 transformers==4.57.1을 설치한 뒤 커널을 재시작하고 다시 실행."
        ) from error

generation_states = ["pending", "error"] if RETRY_ERRORS else ["pending"]
active_ids = set(map(tuple, tasks[KEYS + ["condition"]].to_numpy()))
has_generation = any(
    row["model"] in RUN_MODELS and row["condition"] != "image_only"
    and row["generation_status"] in generation_states
    and tuple(row[c] for c in KEYS + ["condition"]) in active_ids
    for row in result_df.to_dict("records")
)
if has_generation and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 찾지 못했음. 기존에 모델을 실행했던 GPU 커널을 선택.")

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "transformers:", transformers.__version__, "dtype:", DTYPE)


def model_device(model):
    device = model.get_input_embeddings().weight.device
    if device.type == "meta":
        device = next(p.device for p in model.parameters() if p.device.type != "meta")
    return device


def load_model(model_name, revision):
    if model_name == "qwen3":
        from transformers import Qwen3VLForConditionalGeneration
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            MODEL_IDS[model_name], revision=revision, torch_dtype=torch.float16, device_map={"": 0})
        processor = AutoProcessor.from_pretrained(MODEL_IDS[model_name], revision=revision)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_IDS[model_name], revision=revision, trust_remote_code=True,
            torch_dtype=DTYPE, device_map="auto")
        processor = AutoProcessor.from_pretrained(
            MODEL_IDS[model_name], revision=revision, trust_remote_code=True)
    model.eval()
    return model, processor


def generation_kwargs(model_name):
    length_argument = {"qwen3": "max_new_tokens", "hyperclovax": "max_length"}[model_name]
    return {length_argument: MAX_NEW_TOKENS_BY_MODEL[model_name], "do_sample": False}


def generate_one(model_name, model, processor, image_path, prompt, keyword_context=""):
    image_content = {"type": "image", "image": str(image_path)}
    if model_name == "hyperclovax":
        image_content["filename"] = image_path.name
    content = [image_content, {"type": "text", "text": prompt}]
    if keyword_context:
        content.append({"type": "text", "text": keyword_context})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    inputs = inputs.to(model_device(model))
    if model_name == "qwen3":
        inputs.pop("token_type_ids", None)
    else:
        image_tensors = inputs.get("pixel_values_images")
        if image_tensors is None or not any(len(group) > 0 for group in image_tensors):
            raise ValueError("HyperCLOVAX에 이미지가 전달되지 않았음. processor 입력 확인 필요.")
    with torch.inference_mode():

        output_ids = model.generate(**inputs, **generation_kwargs(model_name))
    if hasattr(output_ids, "sequences"):
        output_ids = output_ids.sequences

    if model_name == "qwen3":
        generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    else:

        input_ids = inputs["input_ids"]
        includes_prompt = (output_ids.shape[1] >= input_ids.shape[1]
                           and torch.equal(output_ids[:, :input_ids.shape[1]], input_ids))
        generated_ids = output_ids[:, input_ids.shape[1]:] if includes_prompt else output_ids
    token_count = int(generated_ids.shape[1])
    limit = MAX_NEW_TOKENS_BY_MODEL[model_name]
    if token_count > limit:
        raise ValueError(f"생성 토큰 수 {token_count}개가 설정한 상한 {limit}개를 초과함. 길이 설정 확인 필요.")
    raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    alt = re.sub(r"\s+", " ", raw).strip()
    if not alt:
        raise ValueError("모델이 빈 문장을 출력했음.")

    return alt, raw, token_count


실행 Python: c:\Users\user\miniconda3\envs\capstone_alt\python.exe
Transformers 버전: 4.57.1
GPU: NVIDIA GeForce RTX 5070
torch: 2.7.1+cu128 transformers: 4.57.1 dtype: torch.bfloat16


## 7. 대체텍스트 생성

두 모델을 순서대로 실행하고 조건별 결과를 저장한다. 같은 설정으로 재실행하면 미완료 조건부터 이어서 실행한다.


In [8]:
def check_qwen_english(english):
    return validate_qwen_english(english)

def environment_info(model_name):
    names = ["torch", "transformers", "accelerate", "timm", "einops", "Pillow"]
    info = {}
    for name in names:
        try:
            info[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            info[name] = "missing"
    return dict(info, dtype=str(torch.float16 if model_name == "qwen3" else DTYPE))


def run_model(model_name, df):
    statuses = ["pending", "error"] if RETRY_ERRORS else ["pending"]
    candidates = df.index[(df["model"] == model_name) & df["condition"].ne("image_only")
                          & df["generation_status"].isin(statuses)].tolist()

    active_ids = set(map(tuple, tasks[KEYS + ["condition"]].to_numpy()))
    candidates = [i for i in candidates if tuple(df.loc[i, KEYS + ["condition"]]) in active_ids]
    if not candidates:
        print(model_name, "새로 생성할 조건이 없음.")
        return df

    previous_info = manifest["models"].get(model_name)
    environment = environment_info(model_name)
    if previous_info and previous_info["environment"] != environment:
        raise ValueError(f"{model_name}: 저장된 실행 환경과 다름. 기존 환경을 사용하거나 OUTPUT_DIR을 새 폴더로 변경.")
    revision = previous_info["resolved_revision"] if previous_info else MODEL_REVISIONS[model_name]
    print(f"{model_name} 불러오는 중. 생성 조건: {len(candidates)}개")
    model = processor = None
    try:
        model, processor = load_model(model_name, revision)
        resolved_revision = getattr(model.config, "_commit_hash", None) or revision
        manifest["models"][model_name] = {"model_id": MODEL_IDS[model_name],
                                        "resolved_revision": resolved_revision, "environment": environment}
        atomic_json(manifest, MANIFEST_JSON)
        for i in tqdm(candidates, desc=model_name):
            started = time.perf_counter()
            df.at[i, "raw_output"] = ""
            df.at[i, "generated_token_count"] = ""
            df.at[i, "translation_error"] = ""
            df.at[i, "translated_at"] = ""
            df.at[i, "deepl_sdk_version"] = ""
            df.at[i, "deepl_model_type_used"] = ""
            for column in ("deepl_request_xml", "deepl_response_xml", "protected_names_json", "translation_method"):
                df.at[i, column] = ""
            df.at[i, "generation_source"] = "vlm"
            df.at[i, "english_check"] = ""
            df.at[i, "hit_token_limit"] = ""
            try:
                image_path = PROJECT_ROOT / df.at[i, "image_path"]
                alt, raw, token_count = generate_one(model_name, model, processor, image_path,
                                                     df.at[i, "prompt"], df.at[i, "keyword_context"])
                df.at[i, "raw_output"] = raw
                df.at[i, "generated_token_count"] = token_count
                df.at[i, "hit_token_limit"] = str(token_count >= MAX_NEW_TOKENS_BY_MODEL[model_name])
                if model_name == "qwen3":
                    english = parse_qwen_english_only(raw)
                    english = normalize_qwen_english(english, json.loads(df.at[i, "keywords_json"]))
                    df.at[i, "english_check"] = check_qwen_english(english)
                    df.at[i, "generated_alt_ENG"] = english
                    df.at[i, "generated_alt_KOR_direct"] = ""
                    df.at[i, "generated_alt"] = ""
                    df.at[i, "translation_status"] = "pending"
                else:
                    df.at[i, "generated_alt"] = alt
                    df.at[i, "generated_alt_KOR_direct"] = alt
                    df.at[i, "translation_status"] = "not_required"
                df.at[i, "generation_status"] = "ok"
                df.at[i, "error"] = ""
                df.at[i, "model_revision"] = resolved_revision
            except Exception as error:
                df.at[i, "generation_status"] = "error"
                df.at[i, "generated_alt"] = ""
                df.at[i, "generated_alt_ENG"] = ""
                df.at[i, "generated_alt_KOR_direct"] = ""
                df.at[i, "translation_status"] = "not_generated" if model_name == "qwen3" else "not_required"
                df.at[i, "error"] = f"{type(error).__name__}: {error}"
                print(f"\n오류: {df.at[i, 'image_name']} / {df.at[i, 'condition']}: {error}")
                if isinstance(error, torch.cuda.OutOfMemoryError):
                    torch.cuda.empty_cache()
            df.at[i, "elapsed_seconds"] = round(time.perf_counter() - started, 3)
            df.at[i, "generated_at"] = datetime.now(timezone.utc).isoformat()
            save_results(df)
    finally:
        save_results(df)
        del model, processor
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return df


for model_name in RUN_MODELS:
    result_df = run_model(model_name, result_df)

print("\n저장 완료:", LONG_CSV)
print("비교표:", COMPARE_CSV)


qwen3 새로 생성할 조건이 없음.
hyperclovax 새로 생성할 조건이 없음.

저장 완료: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\alt_keywords_long.csv
비교표: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\alt_keywords_comparison.csv


## 8. Qwen 영어 문장 번역

이름 원문을 XML로 보호하고 DeepL로 한국어 번역한다.
이름 ID·원문이 보존되지 않으면 짧은 표식으로 한 번 더 시도한다. 번역 시도별 요청·응답을 기록한다.
실패한 조건은 영어 원문을 유지하며 `translation_error`에 원인을 기록한다.

이름 표식이 태그 밖으로 이동해도 모든 ID와 정확한 표식이 각각 한 번 존재하면 실제 표식 위치에서 복원한다.
재실행 시 저장된 응답을 먼저 확인하고, 복구된 조건은 다시 번역하지 않는다.


In [9]:
TRANSLATION_REPAIR_VERSION = "v6_exact_short_markers_any_position"
TRANSLATION_ATTEMPTS_JSONL = OUTPUT_DIR / "deepl_translation_attempts.jsonl"






def backup_before_translation_retry():
    import shutil
    backup_dir = OUTPUT_DIR / "backup_before_translation_retry_v5"
    backup_dir.mkdir(parents=True, exist_ok=True)
    for path in (LONG_CSV, MANIFEST_JSON, COMPARE_CSV, READABLE_CSV,
                 QWEN_TRANSLATIONS_CSV, VIEWER_HTML):
        destination = backup_dir / path.name
        if path.is_file() and not destination.exists():
            shutil.copy2(path, destination)
    processing = manifest.setdefault("output_processing", {})
    processing["translation_retry"] = {
        "version": TRANSLATION_REPAIR_VERSION,
        "modes": ["literal", "short_marker"],
        "max_requests_per_condition": 2,
        "options": DEEPL_OPTIONS,
        "backup_directory": backup_dir.name,
        "attempt_log": TRANSLATION_ATTEMPTS_JSONL.name,
    }
    atomic_json(manifest, MANIFEST_JSON)


def log_translation_attempt(df, i, method, payload, names_json, response_xml,
                            outcome, error=None):

    # SDK 예외는 원문 대신 종류만 기록.
    record = {
        "time": datetime.now(timezone.utc).isoformat(),
        "identity": {key: str(df.at[i, key]) for key in KEYS + ["condition"]},
        "source_english": str(df.at[i, "generated_alt_ENG"]),
        "method": method, "options": DEEPL_OPTIONS, "outcome": outcome,
        "request_xml": payload, "protected_names_json": names_json,
        "response_xml": response_xml,
        "error_type": type(error).__name__ if error is not None else "",
        "validation_error": str(error) if isinstance(error, NamePreservationError) else "",
    }
    with TRANSLATION_ATTEMPTS_JSONL.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def request_validated_translation(df, i, english, client):
    for mode in ("literal", "short_marker"):
        method = "quoted_labels_xml_" + mode + ("_v6" if mode == "short_marker" else "_v5")
        payload, names = prepare_retry_translation_xml(english, mode)
        names_json = json.dumps(names, ensure_ascii=False)
        df.at[i, "deepl_request_xml"] = payload
        df.at[i, "protected_names_json"] = names_json
        df.at[i, "deepl_response_xml"] = ""
        df.at[i, "translation_method"] = method
        response_xml = ""
        try:
            response = client.translate_text(payload, **DEEPL_OPTIONS)
            response_xml = str(response.text)
            df.at[i, "deepl_response_xml"] = response_xml
            translated = unpack_retry_translation_xml(response_xml, names)
        except NamePreservationError as error:
            log_translation_attempt(df, i, method, payload, names_json, response_xml,
                                    "invalid_response", error)
            if mode == "literal":
                print(f"\n이름 보호 재시도: {df.at[i, 'image_name']} / {df.at[i, 'condition']}")
                continue
            raise
        except Exception as error:
            log_translation_attempt(df, i, method, payload, names_json, response_xml,
                                    "api_error", error)
            raise
        log_translation_attempt(df, i, method, payload, names_json, response_xml, "validated")
        model_type = str(getattr(response, "model_type_used", "") or "")
        return translated, model_type, payload, response_xml, names_json, method


def translate_qwen_results(df, client):
    if not TRANSLATE_QWEN_WITH_DEEPL:
        print("Qwen 영어 생성 결과는 저장됨. 번역하려면 TRANSLATE_QWEN_WITH_DEEPL = True로 설정.")
        return df
    statuses = ["pending", "error"] if RETRY_TRANSLATION_ERRORS else ["pending"]
    active_ids = set(map(tuple, tasks[KEYS + ["condition"]].to_numpy()))
    candidates = df.index[(df["model"] == "qwen3") & df["condition"].ne("image_only")
                          & df["generation_status"].eq("ok")
                          & df["translation_status"].isin(statuses)].tolist()
    candidates = [i for i in candidates if tuple(df.loc[i, KEYS + ["condition"]]) in active_ids]
    if not candidates:
        print("새로 번역할 Qwen 영어 문장이 없음.")
        return df
    if client is None:
        raise RuntimeError("5번 셀에서 DeepL API 연결을 먼저 확인.")
    backup_before_translation_retry()
    print("번역할 조건:", len(candidates), "개 (완료된 번역 유지)")
    sdk_version = manifest["translator"]["sdk_version"]
    cache = {}
    completed = df.loc[(df["model"] == "qwen3") & df["condition"].ne("image_only")
                       & df["generation_status"].eq("ok") & df["translation_status"].eq("translated")]
    for row in completed.to_dict("records"):
        if row["generated_alt_ENG"] and row["generated_alt"].strip():
            cache[row["generated_alt_ENG"]] = (
                row["generated_alt"], row["deepl_model_type_used"], row["deepl_request_xml"],
                row["deepl_response_xml"], row["protected_names_json"], row["translation_method"],
            )
    for i in tqdm(candidates, desc="Qwen → DeepL"):
        english = df.at[i, "generated_alt_ENG"].strip()
        stop_for_account = False
        try:
            if not english:
                raise ValueError("번역할 영어 원문이 비어 있음.")

            if df.at[i, "translation_status"] == "error":
                log_translation_attempt(
                    df, i, df.at[i, "translation_method"], df.at[i, "deepl_request_xml"],
                    df.at[i, "protected_names_json"], df.at[i, "deepl_response_xml"], "saved_error_before_retry",
                )
            cached = english in cache
            if cached:
                translated, model_type, payload, response_xml, names_json, method = cache[english]
            else:
                translated, model_type, payload, response_xml, names_json, method = request_validated_translation(
                    df, i, english, client,
                )
                cache[english] = (translated, model_type, payload, response_xml, names_json, method)
            df.at[i, "generated_alt"] = translated
            df.at[i, "translation_status"] = "translated"
            df.at[i, "translation_error"] = ""
            df.at[i, "translated_at"] = datetime.now(timezone.utc).isoformat()
            df.at[i, "deepl_sdk_version"] = sdk_version
            df.at[i, "deepl_model_type_used"] = model_type
            df.at[i, "deepl_request_xml"] = payload
            df.at[i, "deepl_response_xml"] = response_xml
            df.at[i, "protected_names_json"] = names_json
            df.at[i, "translation_method"] = method
            df.at[i, "generation_source"] = "vlm + deepl_cached_translation" if cached else "vlm + deepl_api"
        except Exception as error:
            df.at[i, "generated_alt"] = ""
            df.at[i, "translation_status"] = "error"
            if isinstance(error, NamePreservationError):
                message = f"NamePreservationError: {error} 영어 원문은 유지."
            else:
                message = f"{type(error).__name__}: DeepL 번역 실패. API 키·문자 한도·네트워크 확인 필요."
            df.at[i, "translation_error"] = message
            print(f"\n번역 오류: {df.at[i, 'image_name']} / {df.at[i, 'condition']}: {message}")
            stop_for_account = type(error).__name__ in {"AuthorizationException", "QuotaExceededException"}
        save_results(df)
        if stop_for_account:
            print("DeepL 계정 오류로 남은 번역 중단. 계정 문제 해결 후 이 셀에서 이어서 번역.")
            break
    return df


result_df = translate_qwen_results(result_df, deepl_client)
save_results(result_df)
print("Qwen 번역 상태:")
print(result_df[result_df["model"] == "qwen3"].groupby(["condition", "translation_status"]).size().to_string())
print("영어·DeepL 번역 확인 파일:", QWEN_TRANSLATIONS_CSV)
print("번역 시도 기록:", TRANSLATION_ATTEMPTS_JSONL)


새로 번역할 Qwen 영어 문장이 없음.
Qwen 번역 상태:
condition   translation_status
1gram       translated            200
2gram       translated            200
3gram       translated            200
image_only  reused_translation    200
영어·DeepL 번역 확인 파일: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\qwen_english_deepl.csv
번역 시도 기록: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\deepl_translation_attempts.jsonl


## 9. 결과 비교

| 비교표 열 | 내용 |
| --- | --- |
| `prev_alt_KOR` | 기존 컬리 alt |
| `alt_image_only` | 기존 이미지 단독 문장 |
| `alt_1gram`, `alt_2gram`, `alt_3gram` | 핵심어를 추가한 최종 한국어 문장 |
| `alt_ENG_1gram` 등 | Qwen 영어 원문 |
| `keywords_1gram` 등 | 모델에 전달한 핵심어 |

`ok`는 생성 완료, `reused`는 기존 문장 사용, `no_keywords`는 핵심어가 없는 조건을 뜻한다.
`all_ngram_ok`는 1·2·3그램의 생성·번역 완료 여부다. 문장의 정확성은 이미지와 대조해 평가한다.
핵심어가 없는 조건은 결측으로 구분한다. 기존 컬리 alt는 참고 비교 문장으로 사용한다.
기존 이미지 단독 문장과 핵심어 추가 문장은 생성 프롬프트가 다르므로 비교 결과에 이를 명시한다.


In [10]:
print(result_df.groupby(["model", "condition", "generation_status"]).size().to_string())
comparison = comparison_table(result_df)
print("기존 컬리 alt: prev_alt_KOR")
print("이미지 단독 생성: alt_image_only")
print("이미지와 핵심어를 넣은 생성: alt_1gram, alt_2gram, alt_3gram")
print("\n1·2·3그램이 모두 생성된 행:", int(comparison["all_ngram_ok"].sum()))


errors = result_df.loc[result_df["generation_status"].eq("error"),
                       ["model", "image_name", "condition", "keywords", "error", "raw_output"]].copy()
if not errors.empty:
    print("빈칸 중 생성 오류인 조건: 아래 raw_output의 영어 설명과 이름 표기를 확인.")
    with pd.option_context("display.max_colwidth", 600):
        display(errors.head(10))

translation_errors = result_df.loc[
    result_df["model"].eq("qwen3") & result_df["translation_status"].eq("error"),
    ["image_name", "condition", "generated_alt_ENG", "translation_error", "deepl_response_xml"],
].copy()
if not translation_errors.empty:
    print("영어는 생성됐지만 한국어 번역이 비어 있는 조건:")
    with pd.option_context("display.max_colwidth", 600):
        display(translation_errors.head(10))


model        condition   generation_status
hyperclovax  1gram       ok                   300
             2gram       no_keywords            4
                         ok                   296
             3gram       no_keywords           34
                         ok                   266
             image_only  reused               300
qwen3        1gram       ok                   200
             2gram       ok                   200
             3gram       ok                   200
             image_only  reused               200
기존 컬리 alt: prev_alt_KOR
이미지 단독 생성: alt_image_only
이미지와 핵심어를 넣은 생성: alt_1gram, alt_2gram, alt_3gram

1·2·3그램이 모두 생성된 행: 466


In [11]:
import os

column_labels = {
    "category": "카테고리", "model": "모델", "product_id": "상품ID", "image_order": "이미지순서",
    "image_path": "이미지경로", "prev_alt_KOR": "기존 컬리 alt", "alt_image_only": "이미지만 넣은 결과",
    "alt_1gram": "핵심어 추가 1그램", "alt_2gram": "핵심어 추가 2그램", "alt_3gram": "핵심어 추가 3그램",
    "status_image_only": "이미지 단독 상태", "status_1gram": "1그램 상태", "status_2gram": "2그램 상태", "status_3gram": "3그램 상태",
}
if not INCLUDE_IMAGE_ONLY:
    raise ValueError("세 종류 비교 화면을 만들려면 INCLUDE_IMAGE_ONLY=True로 설정.")
readable = comparison[list(column_labels)].rename(columns=column_labels)
atomic_csv(readable, READABLE_CSV)
with pd.option_context("display.max_colwidth", None):
    display(readable.drop(columns=["이미지경로", "이미지 단독 상태", "1그램 상태", "2그램 상태", "3그램 상태"]).head(10))

viewer_template = '<!doctype html>\n<html lang="ko">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<meta name="color-scheme" content="light dark">\n<meta http-equiv="Content-Security-Policy" content="default-src \'none\'; img-src \'self\' blob: data:; style-src \'unsafe-inline\'; script-src \'unsafe-inline\'; connect-src \'none\'; object-src \'none\'; base-uri \'self\'">\n<title>대체텍스트 비교</title>\n<style>\n  :root { color-scheme: light dark; --bg: light-dark(#f6f6f7,#191a1c); --surface: light-dark(#fff,#222426); --text: light-dark(#202124,#f1f1f2); --muted: light-dark(#606368,#b9bdc3); --line: light-dark(#dedfe2,#474a50); --soft: light-dark(#f1f2f4,#303338); --active: light-dark(#2b2d31,#e9eaee); --active-text: light-dark(#fff,#202124); --error: light-dark(#8a2727,#ffb9b9); }\n  * { box-sizing: border-box; }\n  body { margin: 0; background: var(--bg); color: var(--text); font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif; font-size: 15px; line-height: 1.65; }\n  main { max-width: 1240px; padding: 26px 24px 32px; margin: auto; }\n  h1 { font-size: 24px; margin: 0 0 5px; font-weight: 600; }\n  h2 { font-size: 17px; margin: 0; font-weight: 600; }\n  p { margin: 0; }\n  button, select { font: inherit; color: inherit; border: 1px solid var(--line); background: var(--surface); border-radius: 5px; }\n  button { padding: 8px 13px; cursor: pointer; min-height: 42px; }\n  button:hover { background: var(--soft); }\n  button:disabled { opacity: .45; cursor: default; }\n  select { padding: 8px 10px; width: 100%; min-width: 0; min-height: 42px; }\n  .muted, .caption { color: var(--muted); }\n  .caption { font-size: 13px; }\n  .header { display: flex; gap: 18px; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; }\n  .actions { display: flex; gap: 8px; flex-wrap: wrap; }\n  .filters { display: grid; grid-template-columns: 180px 200px minmax(200px,1fr); gap: 14px; margin: 22px 0 14px; }\n  label { display: block; font-size: 13px; margin-bottom: 5px; }\n  .navigation { display: flex; align-items: center; gap: 10px; margin-bottom: 17px; flex-wrap: wrap; }\n  #position { font-variant-numeric: tabular-nums; min-width: 48px; text-align: center; }\n  #dataset-count { margin-left: auto; }\n  .photo-row { display: grid; grid-template-columns: 310px minmax(0,1fr); gap: 22px; margin-bottom: 20px; }\n  .photo { min-height: 218px; display: flex; justify-content: center; align-items: center; background: var(--surface); border: 1px solid var(--line); border-radius: 6px; padding: 14px; }\n  #product-photo { width: 100%; height: 220px; object-fit: contain; }\n  [hidden] { display: none !important; }\n  #photo-missing { text-align: center; color: var(--muted); font-size: 13px; }\n  #photo-missing p + p { margin-top: 10px; }\n  .metadata { padding-top: 4px; min-width: 0; }\n  .metadata-title { font-size: 18px; font-weight: 600; margin-bottom: 8px; }\n  .metadata dl { display: grid; grid-template-columns: 78px minmax(0,1fr); gap: 5px 10px; margin: 0 0 14px; }\n  .metadata dt { color: var(--muted); }\n  .metadata dd { margin: 0; overflow-wrap: anywhere; }\n  .reading-guide { border-top: 1px solid var(--line); padding-top: 12px; }\n  .reading-guide ol { margin: 6px 0 0; padding-left: 22px; }\n  .comparisons { display: grid; grid-template-columns: repeat(3,minmax(0,1fr)); gap: 14px; align-items: stretch; }\n  .text-panel { background: var(--surface); border: 1px solid var(--line); border-radius: 6px; padding: 18px; min-width: 0; }\n  .text-panel .caption { margin-top: 5px; }\n  .alt-text { font-size: 16px; line-height: 1.8; margin-top: 18px; white-space: pre-wrap; overflow-wrap: anywhere; }\n  .empty { color: var(--muted); }\n  .ngram-tabs { display: flex; gap: 5px; margin-top: 12px; }\n  .ngram-tabs button { flex: 1; padding: 7px 6px; font-size: 13px; }\n  .ngram-tabs button[aria-pressed="true"] { background: var(--active); color: var(--active-text); border-color: var(--active); }\n  .condition-status { font-size: 13px; color: var(--muted); margin-top: 12px; }\n  details { margin-top: 16px; border-top: 1px solid var(--line); padding-top: 11px; }\n  summary { color: var(--muted); cursor: pointer; font-size: 13px; }\n  #keywords { font-size: 14px; margin-top: 8px; overflow-wrap: anywhere; }\n  #message { margin: 12px 0; color: var(--error); }\n  .bottom-note { margin-top: 16px; font-size: 13px; color: var(--muted); }\n  input[type="file"] { display: none; }\n  @media(max-width:960px) { .filters { grid-template-columns: 1fr 1fr; } .filters .record-field { grid-column: 1/-1; } .comparisons { grid-template-columns: 1fr; } .photo-row { grid-template-columns: 260px minmax(0,1fr); } }\n  @media(max-width:590px) { main { padding: 18px 14px 26px; } .header { gap: 12px; } .actions { width: 100%; } .actions button { flex: 1; } .photo-row { grid-template-columns: 1fr; gap: 12px; } .filters { gap: 10px; margin-top: 17px; } #dataset-count { width: 100%; margin-left: 0; } .text-panel { padding: 15px; } }\n</style>\n</head>\n<body>\n<main id="alt-review">\n  <header class="header">\n    <div><h1>대체텍스트 비교</h1><p class="muted">같은 이미지에서 기존 alt, 이미지 단독 생성, 핵심어 추가 생성을 비교.</p></div>\n    <div class="actions">\n      <button id="load-csv" type="button">다른 비교 CSV 열기</button>\n      <button id="choose-images" type="button">이미지 폴더 선택</button>\n    </div>\n  </header>\n  <input id="csv-file" type="file" accept=".csv,text/csv">\n  <input id="image-folder" type="file" webkitdirectory multiple>\n  <p id="message" role="alert" hidden></p>\n  <div class="filters">\n    <div><label for="category-filter">카테고리</label><select id="category-filter"></select></div>\n    <div><label for="model-filter">모델</label><select id="model-filter"></select></div>\n    <div class="record-field"><label for="record-select">비교할 이미지</label><select id="record-select"></select></div>\n  </div>\n  <nav class="navigation" aria-label="이미지 이동">\n    <button id="previous" type="button">이전</button><span id="position" aria-live="polite"></span><button id="next" type="button">다음</button>\n    <span class="caption" id="dataset-count"></span>\n  </nav>\n  <div class="photo-row">\n    <div class="photo">\n      <img id="product-photo" alt="" hidden>\n      <div id="photo-missing"><p>상품 이미지</p><p>이 파일을 카테고리 폴더들이 있는<br>폴더에 두고 열거나,<br>위의 ‘이미지 폴더 선택’을 눌러 주세요.</p></div>\n    </div>\n    <div class="metadata">\n      <p class="metadata-title" id="record-title"></p>\n      <dl><dt>상품 ID</dt><dd id="product-id"></dd><dt>모델</dt><dd id="model-name"></dd><dt>이미지 파일</dt><dd id="image-name"></dd></dl>\n      <div class="reading-guide"><p>확인 순서</p><ol><li>이미지의 대상과 문장 내용이 맞는지 확인.</li><li>핵심어를 넣었을 때 중요한 정보가 더 잘 설명되는지 확인.</li><li>비문, 광고 표현, 이미지로 확인할 수 없는 추측이 있는지 확인.</li></ol></div>\n    </div>\n  </div>\n  <section class="comparisons" aria-label="세 가지 대체텍스트 비교">\n    <article class="text-panel"><h2>기존 컬리 alt</h2><p class="caption">사이트에 원래 있던 대체텍스트.</p><p class="alt-text" id="existing-alt"></p></article>\n    <article class="text-panel"><h2>이미지만 넣은 결과</h2><p class="caption">처음 모델 비교 때 생성한 기존 문장. 다시 생성하지 않음.</p><p class="alt-text" id="image-only-alt"></p><p class="condition-status" id="image-only-status"></p></article>\n    <article class="text-panel"><h2>핵심어를 넣은 결과</h2><p class="caption">같은 이미지와 추출한 핵심어로 생성한 문장.</p>\n      <div class="ngram-tabs" role="group" aria-label="N-gram 조건">\n        <button type="button" class="ngram-button" data-n="1" aria-pressed="false">1그램</button>\n        <button type="button" class="ngram-button" data-n="2" aria-pressed="true">2그램</button>\n        <button type="button" class="ngram-button" data-n="3" aria-pressed="false">3그램</button>\n      </div>\n      <p class="alt-text" id="keyword-alt"></p><p class="condition-status" id="keyword-status"></p>\n      <details><summary>사용한 핵심어 보기</summary><p id="keywords"></p></details>\n    </article>\n  </section>\n  <p class="bottom-note">‘생성 완료’는 문장이 출력됐다는 뜻. 품질 점수는 이미지와 대조해서 판단해야 함. 기존 컬리 alt도 정답 설명으로 간주하지 않음.</p>\n  <div class="actions" style="margin-top:16px"><button id="download-csv" type="button">정리된 비교표 CSV 저장</button><span class="caption" id="source-label"></span></div>\n</main>\n<script id="comparison-data" type="application/json">__COMPARISON_DATA__</script>\n<script id="image-path-prefix" type="application/json">__IMAGE_PATH_PREFIX__</script>\n<script>\n(() => {\n  \'use strict\';\n  const $ = id => document.getElementById(id);\n  const categoryLabels = {bakery:\'베이커리\', convenience_meals:\'간편식\', fruits_nuts_rice:\'과일·견과·쌀\', vegetables:\'채소\'};\n  const modelLabels = {qwen3:\'Qwen3-VL 2B\', hyperclovax:\'HyperCLOVAX 3B\'};\n  const aliases = {\'카테고리\':\'category\',\'모델\':\'model\',\'상품ID\':\'product_id\',\'이미지순서\':\'image_order\',\'이미지경로\':\'image_path\',\'기존 컬리 alt\':\'prev_alt_KOR\',\'이미지만 넣은 결과\':\'alt_image_only\',\'핵심어 추가 1그램\':\'alt_1gram\',\'핵심어 추가 2그램\':\'alt_2gram\',\'핵심어 추가 3그램\':\'alt_3gram\',\'이미지 단독 상태\':\'status_image_only\',\'1그램 상태\':\'status_1gram\',\'2그램 상태\':\'status_2gram\',\'3그램 상태\':\'status_3gram\'};\n  let rows = JSON.parse($(\'comparison-data\').textContent);\n  const imagePrefix = JSON.parse($(\'image-path-prefix\').textContent);\n  let visible = [], position = 0, ngram = 2, sourceName = \'첨부한 alt_keywords_comparison.csv\';\n  let imageFiles = [], currentBlob = null;\n  const categoryName = x => categoryLabels[x] || x;\n  const modelName = x => modelLabels[x] || x;\n  const rowKey = r => [r.category,r.model,r.product_id,r.image_order].join(\'|\');\n  function setText(id, value, emptyMessage) {\n    const node = $(id); const empty = !String(value || \'\').trim();\n    node.textContent = empty ? emptyMessage : value; node.classList.toggle(\'empty\',empty);\n  }\n  function populateSelect(node, values, labelFn, allLabel) {\n    node.replaceChildren(); const all = document.createElement(\'option\'); all.value=\'\'; all.textContent=allLabel; node.append(all);\n    for (const value of values) {const option=document.createElement(\'option\'); option.value=value; option.textContent=labelFn(value); node.append(option);}\n  }\n  function initializeFilters() {\n    populateSelect($(\'category-filter\'), [...new Set(rows.map(r=>r.category))], categoryName, \'전체 카테고리\');\n    populateSelect($(\'model-filter\'), [...new Set(rows.map(r=>r.model))], modelName, \'전체 모델\');\n    filterRows();\n  }\n  function filterRows() {\n    const category=$(\'category-filter\').value, model=$(\'model-filter\').value;\n    visible=rows.filter(r=>(!category || r.category===category)&&(!model || r.model===model)); position=0;\n    $(\'record-select\').replaceChildren();\n    visible.forEach((r,i)=>{const option=document.createElement(\'option\'); option.value=String(i); option.textContent=`${categoryName(r.category)} · ${modelName(r.model)} · ${r.product_id} · 이미지 ${r.image_order}`; $(\'record-select\').append(option);});\n    $(\'record-select\').disabled=!visible.length; render();\n  }\n  function selectedStatus(r, condition) {return r[`status_${condition}`] || (r[`alt_${condition}`] ? \'ok\' : \'\');}\n  function statusLabel(status) {return {ok:\'생성 완료\',reused:\'기존 파일에서 가져온 문장\',translation_pending:\'영어 생성 완료 · DeepL 번역 대기\',translation_error:\'영어 생성 완료 · DeepL 번역 오류\',no_keywords:\'핵심어 없음 · 생성 건너뜀\',pending:\'아직 생성하지 않음\',error:\'오류로 생성하지 못함\'}[status] || \'생성 결과 없음\';}\n  function renderKeywords() {\n    const r=visible[position]; const condition=`${ngram}gram`;\n    document.querySelectorAll(\'.ngram-button\').forEach(button=>button.setAttribute(\'aria-pressed\',String(Number(button.dataset.n)===ngram)));\n    if (!r) {setText(\'keyword-alt\',\'\',\'선택된 결과 없음\'); $(\'keyword-status\').textContent=\'\'; $(\'keywords\').textContent=\'\'; return;}\n    const status=selectedStatus(r,condition);\n    let blank=status===\'no_keywords\' ? \'이 이미지의 해당 N-gram 핵심어가 없어 문장을 생성하지 않았음.\' : \'생성된 문장이 없음.\';\n    if([\'translation_pending\',\'translation_error\'].includes(status) && r[`alt_ENG_${condition}`]) blank=\'Qwen 영어 결과 · DeepL 번역 \'+(status===\'translation_error\'?\'오류\':\'대기\')+\'\\n\'+r[`alt_ENG_${condition}`];\n    setText(\'keyword-alt\',r[`alt_${condition}`],blank);\n    $(\'keyword-status\').textContent=`${ngram}그램 · ${statusLabel(status)}`;\n    $(\'keywords\').textContent=r[`keywords_${condition}`] || \'핵심어 기록 없음.\';\n  }\n  function showPhoto(r) {\n    const img=$(\'product-photo\'); img.hidden=true; $(\'photo-missing\').hidden=false;\n    if (currentBlob) {URL.revokeObjectURL(currentBlob);currentBlob=null;}\n    img.removeAttribute(\'src\');\n    if (!r) return;\n    const imagePath=String(r.image_path || `${r.category}/images/kurly_${r.product_id}_${r.image_order}.jpg`).replaceAll(\'\\\\\',\'/\');\n    img.alt=`상품 ${r.product_id}, ${r.image_order}번 원본 이미지`;\n    const suffix=\'/\'+imagePath;\n    let file=imageFiles.find(f=>f.webkitRelativePath.replaceAll(\'\\\\\',\'/\').endsWith(suffix));\n    if (!file) {const name=imagePath.split(\'/\').pop(); const matches=imageFiles.filter(f=>f.name===name);if(matches.length===1)file=matches[0];}\n    img.onload=()=>{img.hidden=false;$(\'photo-missing\').hidden=true;};\n    img.onerror=()=>{img.hidden=true;$(\'photo-missing\').hidden=false;};\n    if(file){currentBlob=URL.createObjectURL(file);img.src=currentBlob;}\n    else if(!imagePath.includes(\'..\') && !imagePath.includes(\':\') && !imagePath.startsWith(\'/\'))img.src=imagePrefix+imagePath;\n  }\n  function render() {\n    const r=visible[position];\n    $(\'position\').textContent=visible.length ? `${position+1} / ${visible.length}` : \'0 / 0\';\n    $(\'previous\').disabled=position===0 || !visible.length;$(\'next\').disabled=position>=visible.length-1 || !visible.length;\n    $(\'record-select\').value=String(position);\n    $(\'dataset-count\').textContent=`현재 ${visible.length}개 이미지·모델 조합 / 전체 ${rows.length}개`;\n    $(\'source-label\').textContent=`자료: ${sourceName}`;\n    $(\'download-csv\').disabled=!visible.length;\n    if(!r){$(\'record-title\').textContent=\'이 조건에 해당하는 결과 없음\'; for(const id of [\'product-id\',\'model-name\',\'image-name\',\'image-only-status\'])$(id).textContent=\'\';setText(\'existing-alt\',\'\',\'선택된 결과 없음\');setText(\'image-only-alt\',\'\',\'선택된 결과 없음\');renderKeywords();showPhoto(null);return;}\n    $(\'record-title\').textContent=`${categoryName(r.category)} · 이미지 ${r.image_order}`;\n    $(\'product-id\').textContent=r.product_id; $(\'model-name\').textContent=modelName(r.model); $(\'image-name\').textContent=r.image_name || String(r.image_path || \'\').split(\'/\').pop();\n    setText(\'existing-alt\',r.prev_alt_KOR,\'기존 alt 값이 비어 있음.\');\n    setText(\'image-only-alt\',r.alt_image_only,\'생성된 문장이 없음.\');\n    $(\'image-only-status\').textContent=statusLabel(selectedStatus(r,\'image_only\'));\n    renderKeywords();showPhoto(r);\n  }\n  function parseCSV(text) {\n    text=text.replace(/^\\uFEFF/,\'\');const matrix=[];let row=[],value=\'\',quoted=false;\n    for(let i=0;i<text.length;i++){const ch=text[i];if(quoted){if(ch===\'"\'&&text[i+1]===\'"\'){value+=\'"\';i++;}else if(ch===\'"\')quoted=false;else value+=ch;}\n      else if(ch===\'"\')quoted=true;else if(ch===\',\'){row.push(value);value=\'\';}else if(ch===\'\\n\'||ch===\'\\r\'){if(ch===\'\\r\'&&text[i+1]===\'\\n\')i++;row.push(value);if(row.some(x=>x!==\'\'))matrix.push(row);row=[];value=\'\';}else value+=ch;}\n    if(quoted)throw new Error(\'CSV 따옴표가 닫히지 않았음.\');\n    if(value!==\'\'||row.length){row.push(value);matrix.push(row);}\n    if(matrix.length<2)throw new Error(\'CSV에 비교할 행이 없음.\');\n    const headers=matrix.shift().map(x=>aliases[x.trim()] || x.trim());\n    const required=[\'category\',\'model\',\'product_id\',\'image_order\',\'prev_alt_KOR\',\'alt_image_only\',\'alt_1gram\',\'alt_2gram\',\'alt_3gram\'];\n    const missing=required.filter(h=>!headers.includes(h));if(missing.length)throw new Error(\'비교표 CSV가 필요함. 빠진 컬럼: \'+missing.join(\', \'));\n    const parsed=matrix.map((cells,i)=>{if(cells.length!==headers.length)throw new Error(`${i+2}행의 컬럼 수가 맞지 않음.`);return Object.fromEntries(headers.map((h,j)=>[h,cells[j]]));});\n    if(new Set(parsed.map(rowKey)).size!==parsed.length)throw new Error(\'같은 이미지·모델 행이 중복되어 있음.\');\n    return parsed;\n  }\n  function csvValue(value){const s=String(value ?? \'\');return \'"\'+s.replaceAll(\'"\',\'""\')+\'"\';}\n  function exportCSV() {\n    const fields=[[\'카테고리\',\'category\'],[\'모델\',\'model\'],[\'상품ID\',\'product_id\'],[\'이미지순서\',\'image_order\'],[\'이미지경로\',\'image_path\'],[\'기존 컬리 alt\',\'prev_alt_KOR\'],[\'이미지만 넣은 결과\',\'alt_image_only\'],[\'핵심어 추가 1그램\',\'alt_1gram\'],[\'핵심어 추가 2그램\',\'alt_2gram\'],[\'핵심어 추가 3그램\',\'alt_3gram\'],[\'이미지 단독 상태\',\'status_image_only\'],[\'1그램 상태\',\'status_1gram\'],[\'2그램 상태\',\'status_2gram\'],[\'3그램 상태\',\'status_3gram\']];\n    const matrix=[fields.map(([label])=>label),...visible.map(r=>fields.map(([,key])=>r[key] || \'\'))];\n    const csv=\'\\uFEFF\'+matrix.map(row=>row.map(csvValue).join(\',\')).join(\'\\r\\n\');\n    const url=URL.createObjectURL(new Blob([csv],{type:\'text/csv;charset=utf-8\'}));const link=document.createElement(\'a\');link.href=url;link.download=\'alt_comparison_readable.csv\';document.body.append(link);link.click();link.remove();setTimeout(()=>URL.revokeObjectURL(url),1000);\n  }\n  $(\'category-filter\').addEventListener(\'change\',filterRows);$(\'model-filter\').addEventListener(\'change\',filterRows);\n  $(\'record-select\').addEventListener(\'change\',()=>{position=Number($(\'record-select\').value);render();});\n  $(\'previous\').addEventListener(\'click\',()=>{if(position>0){position--;render();}});$(\'next\').addEventListener(\'click\',()=>{if(position<visible.length-1){position++;render();}});\n  document.querySelectorAll(\'.ngram-button\').forEach(button=>button.addEventListener(\'click\',()=>{ngram=Number(button.dataset.n);renderKeywords();}));\n  $(\'load-csv\').addEventListener(\'click\',()=>$(\'csv-file\').click());$(\'choose-images\').addEventListener(\'click\',()=>$(\'image-folder\').click());\n  $(\'image-folder\').addEventListener(\'change\',()=>{imageFiles=Array.from($(\'image-folder\').files).filter(f=>/\\.(jpe?g|png|webp)$/i.test(f.name));render();});\n  $(\'csv-file\').addEventListener(\'change\',async()=>{const file=$(\'csv-file\').files[0];if(!file)return;try{const next=parseCSV(await file.text());rows=next;sourceName=file.name;$(\'message\').hidden=true;initializeFilters();}catch(error){$(\'message\').textContent=error.message;$(\'message\').hidden=false;}$(\'csv-file\').value=\'\';});\n  $(\'download-csv\').addEventListener(\'click\',exportCSV);\n  initializeFilters();\n})();\n</script>\n</body>\n</html>\n'
data = json.dumps(comparison.to_dict("records"), ensure_ascii=False, separators=(",", ":"))
data = data.replace("&", r"\u0026").replace("<", r"\u003c").replace(">", r"\u003e")
image_prefix = os.path.relpath(PROJECT_ROOT, OUTPUT_DIR).replace(os.sep, "/").rstrip("/") + "/"
viewer = viewer_template.replace("__COMPARISON_DATA__", data).replace("__IMAGE_PATH_PREFIX__", json.dumps(image_prefix))
temp = VIEWER_HTML.with_suffix(".html.tmp")
temp.write_text(viewer, encoding="utf-8")
temp.replace(VIEWER_HTML)
print("\n비교 화면:", VIEWER_HTML)
print("한글 비교표:", READABLE_CSV)
print("HTML 파일을 브라우저에서 열면 이미지와 세 종류의 문장을 함께 볼 수 있음.")


,카테고리,모델,상품ID,이미지순서,기존 컬리 alt,이미지만 넣은 결과,핵심어 추가 1그램,핵심어 추가 2그램,핵심어 추가 3그램
0,bakery,qwen3,1000128336,1,상품-대표-이미지 1,"금빛 갈색 껍질과 잘게 썬 호두가 듬뿍 들어간 부드럽고 폭신폭신한 속살을 가진 소박한 빵 한 덩이가, 접어 놓은 체크무늬 식탁보 위에 놓여 있다.","황금빛 갈색을 띠고 겉은 바삭하며, 속은 촘촘하면서도 폭신폭신한 호두가 가득 들어간 빵 한 덩이가 얇게 썰려 체크무늬 식탁보 위에 놓여 있다.","황금빛 갈색을 띠고 겉은 바삭하며, 속은 촘촘하면서도 폭신폭신한 빵 한 덩어리가 잘게 다진 견과류가 가득 들어 있어, 얇게 썰린 뒤 흰색과 검은색이 바뀐 체크무늬 식탁보 위에 놓여 있다.","황금빛 갈색을 띠고 겉은 바삭하며, 속은 촘촘하면서도 폭신폭신한 빵 한 덩어리에 잘게 다진 견과류가 가득 들어 있고, 얇게 썰어 흰색과 회색의 체크무늬 식탁보 위에 놓여 있다."
1,bakery,qwen3,1000128336,2,,"황금빛 갈색 껍질과 부드럽고 폭신폭신한 속살에 호두가 듬뿍 박힌 소박하고 정성스럽게 만든 수제 빵 한 덩어리가, 썰려서 체크무늬 식탁보 위에 놓여 있다.","갓 구워낸 소박한 빵 한 덩어리로, 겉은 황금빛 갈색을 띠고 속은 촘촘하면서도 폭신하며 잘게 썬 호두가 가득 들어 있으며, 체크무늬 식탁보 위에 담겨 있다.","갓 구워낸 수제 빵 한 덩어리. 겉은 황금빛 갈색을 띠고 속은 촘촘하면서도 폭신하며, 호두가 듬뿍 박혀 있고, 썰어서 체크무늬 식탁보 위에 올려져 있다.","갓 구워낸 황금빛 호두 빵 한 덩어리가 체크무늬 식탁보 위에 놓여 있는데, 촘촘하면서도 폭신폭신한 속살이 드러나고, 썰어 보니 고소한 호두 속이 엿보인다."
2,bakery,qwen3,1000196053,1,상품-대표-이미지 1,"어두운 색에 갈라진 겉면과 황금빛 갈색 속살을 가진 소박한 타원형 빵 한 덩어리가, 나무 도마 위의 유산지 위에 놓여 있다.","나무 도마 위에 놓인 유산지 위에, 갈라진 짙은 색 껍질과 속이 드러난 녹색 올리브가 보이는 소박한 타원형 빵 한 덩어리.","금빛 갈색을 띤 타원형 빵 한 덩어리가, 갈라진 소박한 겉껍질과 속이 보이는 녹색 올리브를 품고, 나무 도마 위의 유산지 위에 놓여 있다.","금빛 갈색을 띤 타원형 빵으로, 표면은 갈라진 소박한 껍질을 띠고 있으며 속에는 녹색 올리브가 훤히 드러나 있고, 나무 판 위에 놓인 유산지 위에 놓여 있다."
3,bakery,qwen3,1000196053,2,,"황금빛 갈색 껍질을 띤 수제 빵 세 조각의 클로즈업 사진으로, 빵을 잘라내니 녹색 올리브가 가득 들어간 부드럽고 폭신폭신한 속이 드러나 있다.","갓 구워낸 수제 빵 롤 세 개를 클로즈업한 사진으로, 빵 겉면은 황금빛 갈색을 띠고 속에는 녹색 올리브가 선명하게 보이며, 반으로 잘라내어 속이 푹신하고 기공이 많은 모습을 드러내고 있다.","황금빛 갈색 껍질을 띠고 속에는 녹색 올리브가 선명하게 보이는 수제 빵 세 조각의 클로즈업 사진으로, 빵을 잘라내어 속이 푹신하고 기공이 많은 모습을 드러낸 채, 나무 판 위에 놓인 유산지 위에 놓여 있다.","황금빛 갈색 껍질을 띤 수제 빵 세 조각이, 속에는 녹색 올리브가 듬뿍 들어 있고, 나무 판 위에 놓인 유산지 위에 놓여 있는 모습의 클로즈업 사진."
4,bakery,qwen3,1000217187,1,상품-대표-이미지 1,"하얀 타원형 접시 위에는 황금빛 갈색의 둥근 빵 몇 개가 놓여 있고, 그 옆의 밝은 색 천 위에는 같은 빵이 들어 있는 투명한 비닐봉지가 놓여 있다.","하얀 타원형 접시 위에는 황금빛 갈색의 빵 몇 개가 놓여 있고, 그 옆에는 빵이 더 들어 있는 투명한 비닐봉지와 밝은 색 천 위에 ‘""미모모방"" (Mimomofang)’라고 적힌 라벨이 놓여 있다.","하얀 접시 위에는 황금빛 갈색의 둥근 빵 몇 개가 놓여 있고, 그 옆의 밝은 색 식탁보 위에는 같은 빵이 들어 있는 투명한 비닐봉지가 놓여 있다.","하얀 접시 위에는 황금빛 갈색의 둥근 빵 몇 개가 놓여 있고, 그 옆의 밝은 색 식탁보 위에는 같은 빵이 들어 있는 투명한 비닐봉지가 놓여 있다."
5,bakery,qwen3,1000217187,2,,"하얀 접시 위에는 황금빛 갈색을 띠고 부드러워 보이는 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 갈라져 속이 폭신폭신한 모습이 드러나 있다.","하얀 접시 위에는 황금빛 갈색을 띠고 부드러워 보이는 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 갈라져 속이 폭신폭신한 모습이 드러나 있다.","하얀 접시 위에는 황금빛 갈색을 띠고 부드러워 보이는 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 갈라져 속이 폭신하고 푹신한 모습이 드러나 있다.","하얀 접시 위에는 황금빛 갈색을 띠고 부드러워 보이는 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 갈라져 속이 폭신하고 푹신한 모습이 드러나 있다."
6,bakery,qwen3,1000220024,1,상품-대표-이미지 1,"하얀 접시 위에는 황금빛 갈색의 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 잘려 있어 녹아내린 치즈와 햄이 속을 채우고 있는 모습이 드러나 있다.","하얀 접시 위에는 황금빛 갈색의 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 갈라져 안쪽에 녹아내린 치즈와 햄이 드러나 있다.","하얀 접시 위에는 황금빛 갈색의 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 갈라져 있어 안쪽에 녹아내린 치즈와 햄 한 조각이 드러나 있다.","하얀 접시 위에는 황금빛 갈색의 빵 몇 개가 놓여 있는데, 그중 하나는 반으로 잘려 있어 크림 같은 치즈와 햄이 들어간 속이 드러나 있다."
7,bakery,qwen3,1000220024,2,,"하얀 접시 위에는 황금빛 갈색의 빵 6개가 놓여 있는데, 그중 두 개는 반으로 잘려 있어 녹아내린 치즈와 햄이 속을 채우고 있는 것이 보인다. 빵들은 유산지 위에 놓여 있으며, 빵의 부드럽고 기공이 많은 질감이 눈에 띈다.","하얀 접시 위에는 황금빛 갈색을 띠고 부드러워 보이는 빵 몇 개가 놓여 있는데, 그중 두 개는 반으로 잘려 있어 녹아내린 치즈와 햄으로 채워진 속이 드러나 있다.","흰색 접시 위에는 황금빛 갈색의 빵 여섯 개가 놓여 있는데, 그중 두 개는 반으로 잘려 햄과 치즈가 들어간 속이 드러나 있으며, 이 빵들은 유산지 위에 놓여 있다.","흰색 접시 위에는 황금빛 갈색의 빵 여섯 개가 놓여 있는데, 그중 두 개는 반으로 잘려 햄과 치즈가 들어간 속이 드러나 있으며, 이 빵들은 유산지 위에 놓여 있다."
8,bakery,qwen3,1000229955,1,상품-대표-이미지 1,"뚜껑에 파스텔 톤의 산 그림이 그려진 “CHARMANT” 쿠키 상자를 열어보니, 검은색 쟁반 위에 개별 포장된 쿠키 9개가 놓여 있습니다.","""Grains"" 브랜드의 ""CHARMANT"" 테마가 적용된 9종 혼합 쿠키 상자로, 뚜껑에는 파스텔 톤의 산 그림이 그려져 있습니다.","""CHARMANT"" 브랜드의 9종 혼합 쿠키 한 상자로, 파스텔 톤의 산을 모티브로 한 뚜껑이 달린 검은색 트레이에 담겨 있습니다.","‘""CHARMANT""’라고 적힌 분홍색과 파란색의 파스텔 톤 상자 안에는 검은색 쟁반이 들어 있으며, 그 위에는 각각 다른 디자인의 개별 포장된 쿠키 9개가 담겨 있습니다."
9,bakery,qwen3,1000229955,2,,"“CHARMANT” 브랜드의 9종 혼합 쿠키 한 상자가 연한 베이지색 바탕 위에 놓여 있으며, 상자는 열려 있고 뚜껑은 그 옆에 놓여 있습니다. 쿠키들은 투명한 비닐로 개별 포장되어 있으며, 둥근 모양, 사각형, 체크무늬 등 다양한 모양과 색상을 띠고 있습니다.","각각 개별 포장된 9개의 다양한 쿠키가 들어 있는 상자가 밝은 색상의 표면 위에 놓여 있으며, 상자와 쿠키 모두에 ‘""CHARMANT""’라는 브랜드명이 표시되어 있다.","CHARMANT 브랜드의 9종 혼합 쿠키 한 상자로, 둥근 모양, 사각형, 체크무늬 등 다양한 디자인의 쿠키가 들어 있으며, 세련되고 현대적인 패키지에 담겨 있습니다.","CHARMANT 브랜드의 9개가 들어 있는


비교 화면: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\alt_comparison_viewer.html
한글 비교표: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\alt_keywords_comparison_readable.csv
HTML 파일을 브라우저에서 열면 이미지와 세 종류의 문장을 함께 볼 수 있음.


저장 파일:

| 파일 | 내용 |
| --- | --- |
| `alt_keywords_long.csv` | 조건별 문장·핵심어·원본 출력·처리 상태 |
| `alt_keywords_comparison.csv` | 이미지별 비교표 |
| `alt_keywords_comparison_readable.csv` | 한국어 열 이름으로 정리한 비교표 |
| `qwen_english_deepl.csv` | Qwen 영어 원문과 DeepL 번역 |
| `alt_comparison_viewer.html` | 이미지와 문장을 함께 확인하는 비교 화면 |
| `generation_config.json` | 실험 설정·입력 해시·실행 환경 |
| `deepl_translation_attempts.jsonl` | 번역 시도별 요청·응답 |

HTML 파일은 로컬 브라우저에서 연다. 비교 이미지를 찾지 못하면 프로젝트 폴더를 지정한다.


## 10. 실행 결과 점검

저장된 CSV에서 생성 오류·번역 오류·토큰 상한 도달 여부를 확인한다.
토큰 상한에 도달한 문장은 영어 원문 또는 한국어 문장의 끝을 확인한다.
이미지 속 이름과 생성 문장의 이름 표기도 대조한다.


In [12]:
from pathlib import Path
import pandas as pd
from IPython.display import display


DIAGNOSTIC_CSV = Path(globals().get(
    "LONG_CSV",
    Path.cwd() / "keyword_alt_results_preprocessed5_final" / "alt_keywords_long.csv",
))
if not DIAGNOSTIC_CSV.is_file():
    raise FileNotFoundError(f"점검할 결과 파일을 찾지 못했음. DIAGNOSTIC_CSV 확인: {DIAGNOSTIC_CSV}")

diagnostic_df = pd.read_csv(DIAGNOSTIC_CSV, encoding="utf-8-sig", dtype=str, keep_default_na=False)
diagnostic_required = {
    "model", "image_name", "condition", "generation_status", "translation_status",
    "generated_alt_ENG", "generated_alt", "translation_error", "deepl_request_xml",
    "deepl_response_xml", "protected_names_json", "generated_token_count", "hit_token_limit", "raw_output",
}
diagnostic_missing = diagnostic_required - set(diagnostic_df.columns)
if diagnostic_missing:
    raise ValueError(f"현재 버전의 alt_keywords_long.csv가 아님. 없는 열: {sorted(diagnostic_missing)}")

diagnostic_new = diagnostic_df.loc[diagnostic_df["condition"].ne("image_only")].copy()
diagnostic_generation_errors = diagnostic_new.loc[diagnostic_new["generation_status"].eq("error")]
diagnostic_translation_errors = diagnostic_new.loc[
    diagnostic_new["model"].eq("qwen3") & diagnostic_new["translation_status"].eq("error")
].copy()
diagnostic_token_limits = diagnostic_new.loc[
    diagnostic_new["hit_token_limit"].str.strip().str.lower().isin(["true", "1"])
].copy()
print("점검 파일:", DIAGNOSTIC_CSV)
print(diagnostic_df.groupby(["model", "condition", "generation_status"]).size().to_string())
print("\n생성 오류:", len(diagnostic_generation_errors))
print("Qwen 번역 오류:", len(diagnostic_translation_errors))
print("생성 토큰 상한에 도달한 조건:", len(diagnostic_token_limits))
if not diagnostic_token_limits.empty:
    print(diagnostic_token_limits.groupby("model").size().to_string())
print("※ Qwen은 generated_alt_ENG의 영어 문장 끝, HyperCLOVAX는 generated_alt의 한국어 문장 끝을 확인.")
print("※ 토큰 상한 도달은 문장 끊김의 확인 대상이며, 그 자체로 오류 판정은 아님.")

with pd.option_context("display.max_colwidth", None, "display.max_columns", None, "display.max_rows", 30):
    if not diagnostic_translation_errors.empty:
        print("\n[번역 오류 원문·이름 보호 표식·DeepL 반환 내용]")
        display(diagnostic_translation_errors[[
            "image_name", "condition", "generated_alt_ENG", "translation_error",
            "protected_names_json", "deepl_request_xml", "deepl_response_xml",
        ]])
    if not diagnostic_generation_errors.empty:
        print("\n[생성 오류 원문]")
        display(diagnostic_generation_errors[["model", "image_name", "condition", "raw_output"]])
    if not diagnostic_token_limits.empty:
        print("\n[토큰 상한 도달: 원문에서 문장이 끝까지 나왔는지 확인]")
        display(diagnostic_token_limits[[
            "model", "image_name", "condition", "generated_token_count", "generated_alt_ENG", "generated_alt", "raw_output",
        ]])


print("\n이 셀은 저장된 CSV만 읽으며 모델 생성·DeepL 번역을 실행하지 않음.")


if "english_check" in diagnostic_new.columns:
    label_review = diagnostic_new.loc[
        diagnostic_new["english_check"].eq("latin_text_with_parenthetical_labels_review")
    ]
    if not label_review.empty:
        print("\n[이미지와 제품 표기 대조]")
        with pd.option_context("display.max_colwidth", None):
            display(label_review[["image_name", "condition", "generated_alt_ENG", "generated_alt"]])


점검 파일: d:\2026_09_27_alt\keyword_alt_results_preprocessed5_final\alt_keywords_long.csv
model        condition   generation_status
hyperclovax  1gram       ok                   300
             2gram       no_keywords            4
                         ok                   296
             3gram       no_keywords           34
                         ok                   266
             image_only  reused               300
qwen3        1gram       ok                   200
             2gram       ok                   200
             3gram       ok                   200
             image_only  reused               200

생성 오류: 0
Qwen 번역 오류: 0
생성 토큰 상한에 도달한 조건: 0
※ Qwen은 generated_alt_ENG의 영어 문장 끝, HyperCLOVAX는 generated_alt의 한국어 문장 끝을 확인.
※ 토큰 상한 도달은 문장 끊김의 확인 대상이며, 그 자체로 오류 판정은 아님.

이 셀은 저장된 CSV만 읽으며 모델 생성·DeepL 번역을 실행하지 않음.

[이미지와 제품 표기 대조]


,image_name,condition,generated_alt_ENG,generated_alt
1203,kurly_1000030900_1.jpg,3gram,"A white 4kg bag of ""Jeongmi So"" (정미소) ""Bidan Sall"" (비단쌀) rice, with a small bowl of rice above it, set on a gray surface.","회색 바탕 위에 ""Jeongmi So"" (정미소) ""Bidan Sall"" (비단쌀) 쌀이 든 흰색 4kg 봉지가 놓여 있고, 그 위에는 작은 그릇에 담긴 밥이 놓여 있다."
1209,kurly_1000032485_1.jpg,1gram,"A white package of ""Frozen Blueberry"" (블루베리) with a bowl of frozen blueberries above it, set against a light background.","밝은 배경 위에, 그 위에 냉동 블루베리 한 그릇이 놓인 흰색 ""Frozen Blueberry"" (블루베리) 포장지가 놓여 있다."
1210,kurly_1000032485_1.jpg,2gram,"A white package of ""Frozen Blueberry"" (농동 블루베리) with a bowl of frozen blueberries above it, set against a light, textured background.","밝고 질감이 느껴지는 배경 위에, ""Frozen Blueberry"" (농동 블루베리)의 흰색 패키지와 그 위에 얼린 블루베리 한 그릇이 놓여 있다."
1211,kurly_1000032485_1.jpg,3gram,"A white package of ""Frozen Blueberry"" (농동 블루베리) with a small bowl of frozen blueberries above it, set against a light, textured background.","밝고 질감이 느껴지는 배경 위에, ""Frozen Blueberry"" (농동 블루베리)의 흰색 패키지와 그 위에 작은 그릇에 담긴 냉동 블루베리가 놓여 있다."
1498,kurly_5053663_1.jpg,2gram,"A purple bag of ""Ok Rice"" brand ""Jung-yeon"" (야채혼배남 쌀) rice, weighing 10 kg, is displayed with a small bowl of rice and a wooden spoon on a light gray surface.","밝은 회색 바탕 위에 ""Ok Rice"" 브랜드의 ""Jung-yeon"" (야채혼배남 쌀) 쌀이 든 10kg짜리 보라색 봉지와 작은 밥그릇, 나무 숟가락이 놓여 있다."
1545,kurly_5070409_1.jpg,1gram,"A 4kg bag of ""Our Sweet & Soft Rice"" (달보드레쌀) with a bowl of rice beside it, on a light gray surface.","밝은 회색 바탕 위에 ""Our Sweet & Soft Rice"" (달보드레쌀) 4kg 봉지 하나와 그 옆에 밥 한 그릇이 놓여 있다."
1546,kurly_5070409_1.jpg,2gram,"A 4kg bag of ""Our Sweet & Soft Rice"" (달보드레쌀) with a bowl of rice beside it on a light gray surface.","밝은 회색 바탕 위에 ""Our Sweet & Soft Rice"" (달보드레쌀) 4kg 봉지 하나와 그 옆에 밥 한 그릇이 놓여 있다."
1547,kurly_5070409_1.jpg,3gram,"A 4kg bag of ""Our Sweet & Soft Rice"" (달보드레쌀) with a bowl of rice beside it on a light gray surface.","밝은 회색 바탕 위에 ""Our Sweet & Soft Rice"" (달보드레쌀) 4kg 봉지 하나와 그 옆에 밥 한 그릇이 놓여 있다."
